# Biohub Cell Tracking — clean inference notebook

One pass, five models, no unused branches. Everything configurable lives in the
`CFG` cell; the reusable logic is written out as `.py` modules with `%%writefile`
so the forked CPU workers import exactly the code shown here.

| model | role |
| --- | --- |
| `pilkwang/biohub-tracking-support-pack-50ep-v1` | P1: 3D UNet + node transformer (detection + association) and the inference repo |
| `pilkwang/biohub-temporal-unet3d-seed314159-v1` | P2: independent seed blended into the detection and association logits |
| `tweakai/biohub-model-c-combined-primary-v1` | Model C + division GBM: native division candidates and features |
| `tweakai/biohub-public934-source-cardinality-v2` | source-cardinality head: `CONTINUE` vs `DIVIDE(a, b)` per source |
| `tweakai/biohub-motion-corrector-v1` | learned residual on the motion re-link cost |

**Lineage objective.** Detections are linked by an ILP over edge variables
$x_{ij}$, appearance $a_i$, disappearance $b_i$ and division $v_i$:

$$\min_{x,a,b,v}\; -\sum_{(i,j)\in E_0} p_{ij}x_{ij} + 0\sum_i a_i + 1.5\sum_i b_i + 1.0\sum_i v_i,$$

subject to $t_j = t_i + 1$, $\deg^-(i) \le 1$ and $\deg^+(i) \le 2$. Motion-aware
re-linking then rebuilds the frame-to-frame matching with

$$\hat q_i = q_i + 0.5\,(q_i - q_{\mathrm{pred}(i)}), \qquad C_{ij} = \lVert q_j - \hat q_i\rVert_2 + 0.05\lVert q_j - q_i\rVert_2 - p_{ij},$$

solved as a bipartite assignment inside a 6 um gate and then a 10 um recovery
gate, with distances in physical units from the voxel scale
$(1.625, 0.40625, 0.40625)$ um.

In [ ]:
"""CFG — the only place where model links and pipeline parameters are set."""
import json
import os
import time
from pathlib import Path

NOTEBOOK_START = time.monotonic()

INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = INPUT_DIR / "datasets"
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
PKG_DIR = WORKING_DIR / "biohub_pipeline"
PKG_DIR.mkdir(parents=True, exist_ok=True)

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = next(
    (
        path
        for path in (INPUT_DIR / "competitions" / COMPETITION, INPUT_DIR / COMPETITION)
        if path.exists()
    ),
    INPUT_DIR / "competitions" / COMPETITION,
)

CFG = {
    "experiment_tag": "biohub_public934_clear_v1",
    "voxel_scale_um": [1.625, 0.40625, 0.40625],

    # =====================================================================
    # Models. Every checkpoint below is used at inference time; nothing else
    # is loaded. `dataset` is the Kaggle dataset to attach to the notebook.
    # =====================================================================
    "models": {
        # P1 - public 3D UNet + node transformer: detection and association.
        # Also ships the inference repo (scripts/, src/) that is executed.
        "p1": {
            "dataset": "pilkwang/biohub-tracking-support-pack-50ep-v1",
            "root": str(DATASETS_DIR / "pilkwang" / "biohub-tracking-support-pack-50ep-v1"),
            "weights_rel": "weights/unet_transformer/split_0/edge_predictor_best.pth",
            "sha256": "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771",
        },
        # P2 - independent seed of the same architecture, blended into both
        # the detection logits and the association logits (dual-seed ensemble).
        "p2": {
            "dataset": "pilkwang/biohub-temporal-unet3d-seed314159-v1",
            "root": str(DATASETS_DIR / "pilkwang" / "biohub-temporal-unet3d-seed314159-v1"),
            "weights_rel": "unet_transformer/split_0/edge_predictor_best.pth",
            "sha256": "9bac2fa0dadc4a6fc1899e0caf187f4b553e0a7cd90ba1261a68b35ffe9e305f",
        },
        # Model C + division GBM: native division candidates and their features.
        "model_c": {
            "dataset": "tweakai/biohub-model-c-combined-primary-v1",
            "dir": str(DATASETS_DIR / "tweakai" / "biohub-model-c-combined-primary-v1"),
            "checkpoint_rel": "model_c/edge_predictor_best.pth",
        },
        # Source-cardinality V2 head: one CONTINUE option against all retained
        # DIVIDE(a, b) options per source node.
        "source_cardinality": {
            "dataset": "tweakai/biohub-public934-source-cardinality-v2",
            "dir": str(DATASETS_DIR / "tweakai" / "biohub-public934-source-cardinality-v2"),
            "enabled": True,
        },
        # Learned residual on the motion re-link assignment cost.
        "motion_corrector": {
            "dataset": "tweakai/biohub-motion-corrector-v1",
            "path": str(
                DATASETS_DIR / "tweakai" / "biohub-motion-corrector-v1" / "motion_corrector_best.pt"
            ),
            "enabled": True,
            "strength": 1.0,
        },
    },

    # ===================== detection / association ========================
    "detection": {
        "threshold": 0.96875,
        "unet_batch_size": 4,
    },
    "association": {
        "secondary_edge_weight": 0.15,
        "secondary_detection_weight": 0.475,
        "secondary_link_mode": "low_margin_consensus",
        "secondary_mix_temperature": 1.0,
        "secondary_low_margin_max": 0.35,
        "dual_seed_edge_threshold": 0.48,
        # Harmonic forward/reverse mutual-support fusion, in probability space.
        "bidirectional_edge_weight": 0.20,
    },
    "ilp": {
        "use": True,
        "edge_weight": -1.0,
        "appearance": 0.0,
        "disappearance": 1.5,
        "division": 1.0,
    },

    # ===================== output graph post-processing ====================
    "graph": {
        "edge_max_um": 14.0,
        "enforce_next_frame": True,
        "single_parent_repair": True,
        "prune_isolated": True,

        "motion_relink": True,
        "relink_tight_um": 6.0,
        "relink_relaxed_um": 10.0,
        "relink_velocity_weight": 0.5,
        "relink_learned_bonus": 1.0,
        "relink_raw_distance_weight": 0.05,
        # Frames larger than this fall back to the raw ILP edges. The gated,
        # vectorised assignment makes 4000 as cheap as 2600 used to be, so
        # dense videos now keep the learned re-link instead of dropping it.
        "relink_max_frame_nodes": 4000,
        "motion_feature_z_extent_um": 102.375,

        "gap_close": True,
        "gap_close_um": 5.8,
        "gap_density_adaptive": True,
        "gap_density_reference_um": 6.5,
        "gap_density_gain": 0.040,
        "gap_density_max_step_delta_um": 0.125,
        "gap_density_neighbors": 3,
        "gap_reuse_existing": True,
        "gap_reuse_um": 3.2,
        "gap_max_added_frac": 0.05,
        "gap_max_added_abs": 2000,
        "gap_refine_synthetic": True,
        "gap_refine_win_z": 1,
        "gap_refine_win_yx": 3,
        "gap_refine_max_shift_um": 3.2,

        "safe_divisions": True,
        "safe_div_max_um": 4.66,
        "safe_div_sister_max_um": 8.5,
        "safe_div_existing_child_max_um": 7.65,
        "safe_div_sister_score_weight": 0.15,
        "safe_div_frame_frac_cap": 0.0076,
        "safe_div_global_frac_cap": 0.00375,

        "filter_short_tracks": True,
        "min_track_len": 6,
        "keep_division_components": True,

        "linefit_smooth": True,
        "linefit_weight": 0.8,
        "linefit_window": 2,

        "frame_cache_max_frames": 8,
    },
    "division": {
        "combined_decoder": True,
        "model_c_threshold": 0.96,
        "gbm_threshold": 0.65,
        "gbm_rescue_delta": 0.15,
        "gbm_steal_delta": 0.25,
    },

    # ============================== runtime ===============================
    "runtime": {
        "gpu_workers": 2,
        "cpu_workers": 4,
        "cpu_workers_while_predicting": 4,
        "hard_limit_seconds": 11 * 3600 + 50 * 60,
        "finalize_reserve_seconds": 600,
        "min_upgrade_seconds": 60,
        "slice": "",              # e.g. ":1" for a smoke test
        "allow_pip_install": False,
    },
}

CFG["paths"] = {
    "competition_dir": str(COMP_DIR),
    "test_dir": os.environ.get("BIOHUB_TEST_DIR", "").strip() or str(COMP_DIR / "test"),
    "working_dir": str(WORKING_DIR),
    "package_dir": str(PKG_DIR),
    "repo_dir": str(WORKING_DIR / "tracking_repo"),
    "p2_weights_dir": str(WORKING_DIR / "p2_seed_weights"),
    "shard_dir": str(WORKING_DIR / "submission_shards"),
    "model_c_evidence_dir": str(WORKING_DIR / "evidence_model_c"),
    "p1_evidence_dir": str(WORKING_DIR / "evidence_p1"),
    "p2_evidence_dir": str(WORKING_DIR / "evidence_p2"),
    "submission": str(WORKING_DIR / "submission.csv"),
    "run_stats": str(WORKING_DIR / "run_stats.csv"),
    "cfg_json": str(PKG_DIR / "cfg.json"),
}

TEST_DIR = Path(CFG["paths"]["test_dir"])
REPO_DIR = Path(CFG["paths"]["repo_dir"])
SUBMISSION_PATH = Path(CFG["paths"]["submission"])
RUN_STATS_PATH = Path(CFG["paths"]["run_stats"])


def export_cfg() -> None:
    """Publish CFG to the child predictor process and to the .py modules."""
    association = CFG["association"]
    environment = {
        "BIOHUB_SECONDARY_EDGE_WEIGHT": association["secondary_edge_weight"],
        "BIOHUB_SECONDARY_DETECTION_WEIGHT": association["secondary_detection_weight"],
        "BIOHUB_SECONDARY_LINK_MODE": association["secondary_link_mode"],
        "BIOHUB_SECONDARY_MIX_TEMPERATURE": association["secondary_mix_temperature"],
        "BIOHUB_SECONDARY_LOW_MARGIN_MAX": association["secondary_low_margin_max"],
        "BIOHUB_DUAL_SEED_EDGE_THRESHOLD": association["dual_seed_edge_threshold"],
        "BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT": association["bidirectional_edge_weight"],
        "BIOHUB_MODEL_C_CHECKPOINT": str(
            Path(CFG["models"]["model_c"]["dir"]) / CFG["models"]["model_c"]["checkpoint_rel"]
        ),
        "BIOHUB_MODEL_C_EVIDENCE_DIR": CFG["paths"]["model_c_evidence_dir"],
        "BIOHUB_PUBLIC_P1_EVIDENCE_DIR": CFG["paths"]["p1_evidence_dir"],
        "BIOHUB_PUBLIC_P2_EVIDENCE_DIR": CFG["paths"]["p2_evidence_dir"],
        "BIOHUB_REPO_DIR": CFG["paths"]["repo_dir"],
        "BIOHUB_CFG_JSON": CFG["paths"]["cfg_json"],
    }
    for key, value in environment.items():
        os.environ[key] = str(value)
    Path(CFG["paths"]["cfg_json"]).write_text(json.dumps(CFG, indent=2))


export_cfg()

print(f"tag={CFG['experiment_tag']} | test_dir={TEST_DIR} (exists={TEST_DIR.exists()})")
print(
    "models: P1 + P2 dual-seed, Model C + division GBM, source-cardinality V2, "
    "motion corrector"
)

In [ ]:
"""Runtime check, offline dependencies, and materialisation of every artifact."""
import hashlib
import importlib
import importlib.util
import shutil
import subprocess
import sys
import zipfile

import torch

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required; enable the Kaggle GPU accelerator.")
print("CUDA:", torch.cuda.get_device_name(0), "| devices:", torch.cuda.device_count())

# --------------------------------------------------------------------------
# Offline dependencies (wheels ship inside the P1 support pack)
# --------------------------------------------------------------------------
PACKAGE_SPECS = {
    "tracksdata": "tracksdata", "zarr": "zarr>=3.0.10,<4", "pyscipopt": "pyscipopt",
    "geff": "geff>=1.1.3.1.1", "geff_spec": "geff-spec<1.2", "ilpy": "ilpy>=0.5.1",
    "polars": "polars>=1.36", "blosc2": "blosc2", "dask": "dask",
    "imagecodecs": "imagecodecs", "skimage": "scikit-image>=0.24", "pyarrow": "pyarrow",
    "rustworkx": "rustworkx>=0.17.1", "sqlalchemy": "sqlalchemy>=2",
    "numcodecs": "numcodecs>=0.13,<0.16", "donfig": "donfig>=0.8",
    "google_crc32c": "google-crc32c>=1.5", "bidict": "bidict>=0.23.1",
    "psygnal": "psygnal>=0.14", "rich": "rich", "networkx": "networkx>=3.2.1",
    "pydantic": "pydantic>=2.11", "pydantic_core": "pydantic-core",
    "annotated_types": "annotated-types", "typing_extensions": "typing-extensions>=4.13",
    "typing_inspection": "typing-inspection", "markdown_it": "markdown-it-py",
    "pygments": "pygments", "click": "click", "cloudpickle": "cloudpickle",
    "fsspec": "fsspec", "partd": "partd", "locket": "locket", "toolz": "toolz",
    "yaml": "pyyaml", "ndindex": "ndindex", "msgpack": "msgpack", "numexpr": "numexpr",
    "deprecated": "deprecated", "wrapt": "wrapt", "imageio": "imageio", "PIL": "pillow",
    "tifffile": "tifffile", "lazy_loader": "lazy-loader", "tqdm": "tqdm",
    "threadpoolctl": "threadpoolctl",
}
EXTRA_SPECS = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"], "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"], "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13",
                 "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
os.environ.setdefault("POLARS_PREFER_PKG", "32")


def _wheel_dirs() -> list[Path]:
    found = []
    for parent in (INPUT_DIR.glob("*"), INPUT_DIR.glob("*/*"), INPUT_DIR.glob("*/*/*")):
        for child in parent:
            for candidate in (child / "wheels", child):
                if candidate.is_dir() and any(candidate.glob("*.whl")) and candidate not in found:
                    found.append(candidate)
    return found


def _importable(name: str) -> bool:
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


def _runtime_healthy() -> list[str]:
    """Packages that are importable but unusable in this kernel."""
    broken = []
    try:
        import polars as pl
        from polars._plr import PySeries  # noqa: F401

        if not hasattr(pl, "Float16"):
            broken.append("polars")
    except Exception:
        broken.append("polars")
    try:
        import zarr

        if int(str(zarr.__version__).split(".", 1)[0]) < 3:
            broken.append("zarr")
    except Exception:
        broken.append("zarr")
    return broken


def _install(names: list[str]) -> None:
    specs, seen = [], set()
    for name in names:
        for spec in [PACKAGE_SPECS.get(name, name), *EXTRA_SPECS.get(name, [])]:
            if spec.lower() not in seen:
                seen.add(spec.lower())
                specs.append(spec)
    command = [sys.executable, "-m", "pip", "install", "--no-deps", "-q"]
    if {"polars", "zarr"} & set(names):
        command.append("--force-reinstall")
    wheels = _wheel_dirs()
    if wheels:
        offline = [*command, "--no-index"]
        for directory in wheels:
            offline += ["--find-links", str(directory)]
        if subprocess.run([*offline, *specs], text=True, capture_output=True).returncode == 0:
            return
    if CFG["runtime"]["allow_pip_install"]:
        if subprocess.run([*command, *specs], text=True, capture_output=True).returncode == 0:
            return
    raise ImportError(f"Could not install: {names}. Attach the wheels artifact.")


def ensure_dependencies() -> None:
    for _ in range(5):
        broken = [name for name in PACKAGE_SPECS if not _importable(name)]
        broken += [name for name in _runtime_healthy() if name not in broken]
        if not broken:
            return
        _install(broken)
        for name in list(sys.modules):
            if name.split(".")[0] in {n.split(".")[0] for n in broken}:
                sys.modules.pop(name, None)
    raise ImportError("Dependency recovery did not converge.")


ensure_dependencies()


# --------------------------------------------------------------------------
# Artifact resolution by manifest checksum
# --------------------------------------------------------------------------
def _manifest_roots(hint: str):
    yield Path(hint)
    for depth in ("*", "*/*", "*/*/*"):
        for manifest in INPUT_DIR.glob(f"{depth}/ARTIFACT_MANIFEST.json"):
            yield manifest.parent


def resolve_artifact(hint: str, sha256: str) -> Path:
    seen = set()
    for root in _manifest_roots(hint):
        if root in seen or not (root / "ARTIFACT_MANIFEST.json").is_file():
            continue
        seen.add(root)
        try:
            manifest = json.loads((root / "ARTIFACT_MANIFEST.json").read_text())
        except Exception:
            continue
        if str(manifest.get("model", {}).get("weight_sha256", "")) == sha256:
            return root
    raise FileNotFoundError(f"No attached artifact has weight_sha256={sha256} (hint: {hint})")


def _unpack(source_dir: Path, source_zip: Path, destination: Path) -> None:
    if destination.is_symlink() or destination.is_file():
        destination.unlink()
    elif destination.exists():
        shutil.rmtree(destination)
    if source_dir.is_dir():
        shutil.copytree(source_dir, destination)
    elif source_zip.is_file():
        destination.mkdir(parents=True)
        with zipfile.ZipFile(source_zip) as archive:
            archive.extractall(destination)
    else:
        raise FileNotFoundError(f"Missing {source_dir} and {source_zip}")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


# P1: inference repo + primary weights.
P1 = CFG["models"]["p1"]
p1_root = resolve_artifact(P1["root"], P1["sha256"])
_unpack(p1_root / "repo", p1_root / "repo.zip", REPO_DIR)
weights_destination = REPO_DIR / "weights"
if (p1_root / "weights").is_dir():
    if weights_destination.exists() or weights_destination.is_symlink():
        shutil.rmtree(weights_destination, ignore_errors=True)
    try:
        os.symlink(p1_root / "weights", weights_destination, target_is_directory=True)
    except Exception:
        shutil.copytree(p1_root / "weights", weights_destination)
else:
    _unpack(p1_root / "weights", p1_root / "weights.zip", weights_destination)

P1_WEIGHTS_REL = P1["weights_rel"]
for required in (REPO_DIR / "scripts" / "predict_unet_transformer.py", REPO_DIR / P1_WEIGHTS_REL):
    if not required.exists():
        raise FileNotFoundError(f"Incomplete inference repo: {required}")

# P2: independent-seed weights only.
P2 = CFG["models"]["p2"]
p2_root = resolve_artifact(P2["root"], P2["sha256"])
P2_WEIGHTS_ROOT = Path(CFG["paths"]["p2_weights_dir"])
_unpack(p2_root / "weights", p2_root / "weights.zip", P2_WEIGHTS_ROOT)
P2_WEIGHTS = P2_WEIGHTS_ROOT / P2["weights_rel"]
if not P2_WEIGHTS.is_file() or not (P2_WEIGHTS.parent / "config.json").is_file():
    raise FileNotFoundError(f"Missing secondary model files under {P2_WEIGHTS_ROOT}")
if _sha256(P2_WEIGHTS) != P2["sha256"]:
    raise RuntimeError("Secondary model checksum mismatch")
os.environ["BIOHUB_SECONDARY_WEIGHTS"] = str(P2_WEIGHTS)

# Source-cardinality runtime: Kaggle may keep the uploaded ZIP directory layout.
CARDINALITY_DIR = Path(CFG["models"]["source_cardinality"]["dir"])
if not (CARDINALITY_DIR / "source_cardinality_graph_runtime.py").is_file():
    candidates = sorted({
        path.parent
        for path in INPUT_DIR.rglob("source_cardinality_graph_runtime.py")
        if (path.parent / "source_cardinality_head.pt").is_file()
    })
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one source-cardinality artifact, found {candidates}")
    CARDINALITY_DIR = candidates[0]
    CFG["models"]["source_cardinality"]["dir"] = str(CARDINALITY_DIR)

MODEL_C_DIR = Path(CFG["models"]["model_c"]["dir"])
for required in (
    Path(CFG["models"]["motion_corrector"]["path"]),
    MODEL_C_DIR / CFG["models"]["model_c"]["checkpoint_rel"],
    MODEL_C_DIR / "division_gbm_runtime.py",
    MODEL_C_DIR / "model_c_safe_runtime.py",
    CARDINALITY_DIR / "source_cardinality_head.pt",
    CARDINALITY_DIR / "source_cardinality_runtime.py",
    CARDINALITY_DIR / "source_cardinality_graph_runtime.py",
    CARDINALITY_DIR / "export_model_c_native_division_evidence.py",
):
    if not required.exists():
        raise FileNotFoundError(f"Missing helper artifact: {required}")

# Fresh evidence and shard directories for this run.
for key in ("model_c_evidence_dir", "p1_evidence_dir", "p2_evidence_dir", "shard_dir"):
    directory = Path(CFG["paths"][key])
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)

export_cfg()

TEST_STEMS = sorted(
    path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr")
)
if not TEST_STEMS:
    raise FileNotFoundError(f"No test .zarr volumes in {TEST_DIR}")

print("P1:", p1_root)
print("P2:", P2_WEIGHTS)
print("Model C:", MODEL_C_DIR, "| cardinality:", CARDINALITY_DIR)
print(f"Test videos: {len(TEST_STEMS)}")

In [ ]:
%%writefile /kaggle/working/biohub_pipeline/patch_predictor.py
"""In-place patches for the artifact copy of `predict_unet_transformer.py`.

Run as a script (the notebook does this once, before any GPU work):

    BIOHUB_REPO_DIR=/kaggle/working/tracking_repo python patch_predictor.py

Every replacement is asserted to match exactly once, so a silent mismatch
with a different artifact build fails the run instead of degrading it.
"""
from __future__ import annotations

import os
from pathlib import Path

_ps = Path(os.environ["BIOHUB_REPO_DIR"]) / "scripts" / "predict_unet_transformer.py"


def _apply(source: str, replacements, label: str) -> str:
    for index, (old, new) in enumerate(replacements, start=1):
        count = source.count(old)
        if count != 1:
            raise RuntimeError(f"{label} patch {index}: expected 1 match, found {count}")
        source = source.replace(old, new, 1)
    compile(source, str(_ps), "exec")
    return source


_s = _ps.read_text()

# 1. Eight-view (D4) planar detection TTA in place of the default four flips.
_old = """        if cfg.det_tta:
            tta_flips = [(-1,), (-2,), (-2, -1)]
            for dims in tta_flips:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
            for f in range(W):
                det_logits[f] = det_logits[f] / 4"""
_new = """        if cfg.det_tta:
            _nv = 1
            for dims in [(-1,), (-2,), (-2, -1)]:
                imgs_flip = imgs.flip(dims)
                _, det_flip = model.encode(imgs_flip)
                for f in range(W):
                    det_logits[f] = det_logits[f] + det_flip[f].flip(dims)
                del imgs_flip, det_flip
                _nv += 1
            for _k in (1, 3):
                imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))
                _, det_rot = model.encode(imgs_rot)
                for f in range(W):
                    det_logits[f] = det_logits[f] + torch.rot90(det_rot[f], -_k, dims=(-2, -1))
                del imgs_rot, det_rot
                _nv += 1
            imgs_t = imgs.transpose(-1, -2)
            _, det_t = model.encode(imgs_t)
            for f in range(W):
                det_logits[f] = det_logits[f] + det_t[f].transpose(-1, -2)
            del imgs_t, det_t
            _nv += 1
            imgs_at = torch.rot90(imgs, 1, dims=(-2, -1)).transpose(-1, -2)
            _, det_at = model.encode(imgs_at)
            for f in range(W):
                det_logits[f] = det_logits[f] + torch.rot90(det_at[f].transpose(-1, -2), -1, dims=(-2, -1))
            del imgs_at, det_at
            _nv += 1
            for f in range(W):
                det_logits[f] = det_logits[f] / _nv"""
_s = _apply(_s, [(_old, _new)], "detection TTA")

# 2. Calibrated dual-seed ensemble: second temporal model on the same graph.
_ensemble_replacements = [
    ('    downsample: tuple[int, ...] = (1, 4, 4),\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    downsample: tuple[int, ...] = (1, 4, 4),\n    secondary_model: UNetNodeTransformer | None = None,\n    secondary_edge_weight: float = 0.0,\n    secondary_detection_weight: float = 0.0,\n    secondary_link_mode: str = "raw",\n    secondary_mix_temperature: float = 1.0,\n    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'),
    ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        del imgs', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:\n                if cfg.det_tta:\n                    _secondary_nv = 1\n                    for dims in [(-1,), (-2,), (-2, -1)]:\n                        secondary_imgs_flip = imgs.flip(dims)\n                        _, secondary_det_flip = secondary_model.encode(secondary_imgs_flip)\n                        for f in range(W):\n                            secondary_det_logits[f] = (\n                                secondary_det_logits[f] + secondary_det_flip[f].flip(dims)\n                            )\n                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):\n                        secondary_imgs_rot = torch.rot90(imgs, _k, dims=(-2, -1))\n                        _, secondary_det_rot = secondary_model.encode(secondary_imgs_rot)\n                        for f in range(W):\n                            secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                                secondary_det_rot[f], -_k, dims=(-2, -1)\n                            )\n                        del secondary_imgs_rot, secondary_det_rot\n                        _secondary_nv += 1\n                    secondary_imgs_t = imgs.transpose(-1, -2)\n                    _, secondary_det_t = secondary_model.encode(secondary_imgs_t)\n                    for f in range(W):\n                        secondary_det_logits[f] = (\n                            secondary_det_logits[f] + secondary_det_t[f].transpose(-1, -2)\n                        )\n                    del secondary_imgs_t, secondary_det_t\n                    _secondary_nv += 1\n                    secondary_imgs_at = torch.rot90(\n                        imgs, 1, dims=(-2, -1)\n                    ).transpose(-1, -2)\n                    _, secondary_det_at = secondary_model.encode(secondary_imgs_at)\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] + torch.rot90(\n                            secondary_det_at[f].transpose(-1, -2),\n                            -1,\n                            dims=(-2, -1),\n                        )\n                    del secondary_imgs_at, secondary_det_at\n                    _secondary_nv += 1\n                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):\n                    primary_det = det_logits[f]\n                    secondary_det = secondary_det_logits[f]\n                    primary_mean = primary_det.mean()\n                    secondary_mean = secondary_det.mean()\n                    primary_scale = primary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    secondary_scale = secondary_det.float().std(unbiased=False).clamp_min(1e-4)\n                    scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_det_aligned = (\n                        (secondary_det - secondary_mean) * scale_ratio + primary_mean\n                    )\n                    det_logits[f] = (\n                        (1.0 - secondary_detection_weight) * primary_det\n                        + secondary_detection_weight * secondary_det_aligned\n                    )\n\n            del secondary_det_logits\n\n        del imgs'),
    ('            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            raw = edge_logits_pair[0]', '            edge_logits_pair = model.predict_edges(\n                unet_feat_src, unet_feat_tgt,\n                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                p_pos_src, p_pos_tgt,\n                p_mask_src, p_mask_tgt,\n            )  # (1, n_src, n_tgt)\n\n            if secondary_model is not None:\n                if secondary_unet_out is None:\n                    raise RuntimeError("Secondary model is loaded but its feature map is missing")\n                secondary_feat_src = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx], p_coords_src, p_mask_src,\n                )\n                secondary_feat_tgt = secondary_model._index_features(\n                    secondary_unet_out[:, f_idx + 1], p_coords_tgt, p_mask_tgt,\n                )\n                secondary_logits_pair = secondary_model.predict_edges(\n                    secondary_feat_src, secondary_feat_tgt,\n                    p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,\n                    p_pos_src, p_pos_tgt,\n                    p_mask_src, p_mask_tgt,\n                )\n\n                if secondary_link_mode == "raw":\n                    secondary_for_mix = secondary_logits_pair\n                    blend_weight = secondary_edge_weight\n                elif secondary_link_mode in {\n                    "calibrated", "adaptive", "low_margin_consensus"\n                }:\n                    primary_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    primary_scale = edge_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_center = secondary_logits_pair.mean(dim=1, keepdim=True)\n                    secondary_scale = secondary_logits_pair.float().std(\n                        dim=1, keepdim=True, unbiased=False\n                    ).clamp_min(1e-4)\n                    secondary_scale_ratio = (primary_scale / secondary_scale).clamp(0.5, 2.0)\n                    secondary_for_mix = (\n                        (secondary_logits_pair - secondary_center) * secondary_scale_ratio\n                        + primary_center\n                    )\n                    if secondary_link_mode == "calibrated":\n                        blend_weight = secondary_edge_weight\n                    elif secondary_link_mode == "adaptive":\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            secondary_margin = secondary_top2.values[0] - secondary_top2.values[1]\n                            local_weight = (\n                                secondary_edge_weight + secondary_margin - primary_margin\n                            ).clamp(0.15, 0.75)\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            local_weight = torch.where(\n                                same_parent,\n                                torch.maximum(\n                                    local_weight,\n                                    torch.full_like(local_weight, secondary_edge_weight),\n                                ),\n                                local_weight,\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = secondary_edge_weight\n                    else:\n                        if n_src >= 2:\n                            primary_probs = torch.softmax(edge_logits_pair[0], dim=0)\n                            secondary_probs = torch.softmax(secondary_for_mix[0], dim=0)\n                            primary_top2 = torch.topk(primary_probs, k=2, dim=0)\n                            secondary_top2 = torch.topk(secondary_probs, k=2, dim=0)\n                            primary_margin = primary_top2.values[0] - primary_top2.values[1]\n                            same_parent = primary_top2.indices[0].eq(\n                                secondary_top2.indices[0]\n                            )\n                            uncertainty = (\n                                (secondary_low_margin_max - primary_margin)\n                                / secondary_low_margin_max\n                            ).clamp(0.0, 1.0)\n                            local_weight = secondary_edge_weight * uncertainty\n                            local_weight = torch.where(\n                                same_parent,\n                                local_weight,\n                                torch.zeros_like(local_weight),\n                            )\n                            blend_weight = local_weight.view(1, 1, -1)\n                        else:\n                            blend_weight = 0.0\n                else:\n                    raise ValueError(f"Unsupported secondary link mode: {secondary_link_mode}")\n\n                edge_logits_pair = (\n                    (1.0 - blend_weight) * edge_logits_pair\n                    + blend_weight * secondary_for_mix\n                )\n                if secondary_mix_temperature != 1.0:\n                    mixed_center = edge_logits_pair.mean(dim=1, keepdim=True)\n                    edge_logits_pair = mixed_center + (\n                        edge_logits_pair - mixed_center\n                    ) / secondary_mix_temperature\n\n            raw = edge_logits_pair[0]'),
    ('        del unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n'),
    ('    model, window_size, downsample = load_model(weights_path, device)\n    print(', '    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None\n    secondary_weights_text = os.environ.get("BIOHUB_SECONDARY_WEIGHTS", "").strip()\n    secondary_edge_weight = float(os.environ.get("BIOHUB_SECONDARY_EDGE_WEIGHT", "0"))\n    secondary_detection_weight = float(\n        os.environ.get("BIOHUB_SECONDARY_DETECTION_WEIGHT", "0")\n    )\n    secondary_link_mode = os.environ.get("BIOHUB_SECONDARY_LINK_MODE", "raw").strip()\n    secondary_mix_temperature = float(\n        os.environ.get("BIOHUB_SECONDARY_MIX_TEMPERATURE", "1")\n    )\n    secondary_low_margin_max = float(\n        os.environ.get("BIOHUB_SECONDARY_LOW_MARGIN_MAX", "0.2")\n    )\n    edge_candidate_threshold = float(\n        os.environ.get("BIOHUB_DUAL_SEED_EDGE_THRESHOLD", str(cfg.threshold))\n    )\n    if secondary_weights_text:\n        if not 0.0 < secondary_edge_weight < 1.0:\n            raise ValueError("BIOHUB_SECONDARY_EDGE_WEIGHT must be strictly between 0 and 1")\n        if not 0.0 <= secondary_detection_weight < 1.0:\n            raise ValueError(\n                "BIOHUB_SECONDARY_DETECTION_WEIGHT must be in the half-open interval [0, 1)"\n            )\n        if secondary_link_mode not in {\n            "raw", "calibrated", "adaptive", "low_margin_consensus"\n        }:\n            raise ValueError(\n                "BIOHUB_SECONDARY_LINK_MODE must be raw, calibrated, adaptive, "\n                "or low_margin_consensus"\n            )\n        if not 0.5 <= secondary_mix_temperature <= 2.0:\n            raise ValueError("BIOHUB_SECONDARY_MIX_TEMPERATURE must be in [0.5, 2.0]")\n        if not 0.0 < edge_candidate_threshold < 1.0:\n            raise ValueError("BIOHUB_DUAL_SEED_EDGE_THRESHOLD must be strictly between 0 and 1")\n        if not 0.0 < secondary_low_margin_max <= 1.0:\n            raise ValueError("BIOHUB_SECONDARY_LOW_MARGIN_MAX must be in (0, 1]")\n        secondary_model, secondary_window_size, secondary_downsample = load_model(\n            Path(secondary_weights_text), device,\n        )\n        if secondary_window_size != window_size or secondary_downsample != downsample:\n            raise ValueError(\n                "Primary and secondary models have incompatible inference grids: "\n                f"primary=(window={window_size}, downsample={downsample}), "\n                f"secondary=(window={secondary_window_size}, downsample={secondary_downsample})"\n            )\n        cfg.threshold = edge_candidate_threshold\n        print(\n            f"Secondary model: {secondary_weights_text} | "\n            f"edge weight={secondary_edge_weight:.3f} | "\n            f"detection weight={secondary_detection_weight:.3f} | "\n            f"link mode={secondary_link_mode} | "\n            f"temperature={secondary_mix_temperature:.3f} | "\n            f"low-margin max={secondary_low_margin_max:.3f} | "\n            f"edge threshold={cfg.threshold:.3f}",\n            flush=True,\n        )\n\n    print('),
    ('                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n            )', '                unet_batch_size=unet_batch_size,\n                downsample=downsample,\n                secondary_model=secondary_model,\n                secondary_edge_weight=secondary_edge_weight,\n                secondary_detection_weight=secondary_detection_weight,\n                secondary_link_mode=secondary_link_mode,\n                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )'),
]
_s = _apply(_s, _ensemble_replacements, "dual-seed")

# 3. Bidirectional harmonic-probability association fusion.
_bi_old = '''            edge_logits_pair = model.predict_edges(
                unet_feat_src, unet_feat_tgt,
                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,
                p_pos_src, p_pos_tgt,
                p_mask_src, p_mask_tgt,
            )  # (1, n_src, n_tgt)

            if secondary_model is not None:
'''
_bi_new = '''            edge_logits_pair = model.predict_edges(
                unet_feat_src, unet_feat_tgt,
                p_coords_src * ds_arr_t, p_coords_tgt * ds_arr_t,
                p_pos_src, p_pos_tgt,
                p_mask_src, p_mask_tgt,
            )  # (1, n_src, n_tgt)

            _bidirectional_weight = float(
                os.environ.get("BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT", "0")
            )
            if _bidirectional_weight > 0.0:
                reverse_logits_native = model.predict_edges(
                    unet_feat_tgt, unet_feat_src,
                    p_coords_tgt * ds_arr_t, p_coords_src * ds_arr_t,
                    p_pos_tgt, p_pos_src,
                    p_mask_tgt, p_mask_src,
                )  # (1, n_tgt, n_src)
                reverse_logits_pair = reverse_logits_native.transpose(1, 2)

                forward_center = edge_logits_pair.mean(dim=1, keepdim=True)
                forward_scale = edge_logits_pair.float().std(
                    dim=1, keepdim=True, unbiased=False
                ).clamp_min(1e-4)
                reverse_center = reverse_logits_pair.mean(dim=1, keepdim=True)
                reverse_scale = reverse_logits_pair.float().std(
                    dim=1, keepdim=True, unbiased=False
                ).clamp_min(1e-4)
                reverse_scale_ratio = (forward_scale / reverse_scale).clamp(0.5, 2.0)
                reverse_scale_ratio = reverse_scale_ratio.to(reverse_logits_pair.dtype)
                reverse_aligned = (
                    (reverse_logits_pair - reverse_center) * reverse_scale_ratio
                    + forward_center
                )
                # Biohub 145: require mutual forward/reverse support in probability space.
                # The harmonic mean penalizes a candidate when either temporal direction
                # assigns it very low probability, while calibration preserves the forward
                # logit scale used by the unchanged downstream candidate threshold and ILP.
                forward_prob = torch.softmax(edge_logits_pair.float(), dim=1).clamp_min(1e-8)
                reverse_prob = torch.softmax(reverse_aligned.float(), dim=1).clamp_min(1e-8)
                harmonic_prob = 1.0 / (
                    (1.0 - _bidirectional_weight) / forward_prob
                    + _bidirectional_weight / reverse_prob
                )
                harmonic_prob = harmonic_prob / harmonic_prob.sum(
                    dim=1, keepdim=True
                ).clamp_min(1e-8)
                harmonic_logits = torch.log(harmonic_prob.clamp_min(1e-8))
                harmonic_center = harmonic_logits.mean(dim=1, keepdim=True)
                harmonic_scale = harmonic_logits.std(
                    dim=1, keepdim=True, unbiased=False
                ).clamp_min(1e-4)
                harmonic_scale_ratio = (forward_scale / harmonic_scale).clamp(0.5, 2.0)
                edge_logits_pair = (
                    (harmonic_logits - harmonic_center) * harmonic_scale_ratio
                    + forward_center
                ).to(reverse_aligned.dtype)
                del (
                    reverse_logits_native,
                    reverse_logits_pair,
                    reverse_aligned,
                    forward_prob,
                    reverse_prob,
                    harmonic_prob,
                    harmonic_logits,
                )

            if secondary_model is not None:
'''
_s = _apply(_s, [(_bi_old, _bi_new)], "bidirectional fusion")

# 4. Native Model-C division evidence emitted inside the same GPU pass.
_model_c_predictor_replacements = [('import sys\n', 'import sys\nimport time\n'), ('    secondary_low_margin_max: float = 0.2,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    secondary_low_margin_max: float = 0.2,\n    division_model: UNetNodeTransformer | None = None,\n    division_evidence_path: Path | None = None,\n    division_det_threshold: float = 0.99,\n    division_pool_um: float = 3.0,\n    division_radius_um: float = 20.0,\n    division_topk: int = 16,\n    division_min_probability: float = 0.01,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'), ('    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\n\n    # Running node registry', '    pool_k = pool_kernel_from_um(cfg.pool_kernel_um, voxel_size)\n    division_pool_k = pool_kernel_from_um(division_pool_um, voxel_size)\n    division_ds_arr_t = torch.from_numpy(ds_arr).to(device)\n    original_spacing = np.asarray(ds.scale, dtype=np.float32)\n    division_seen_frames: set[int] = set()\n    division_seen_pairs: set[tuple[int, int]] = set()\n    division_coord_by_t: dict[int, np.ndarray] = {}\n    division_offset: dict[int, tuple[int, int]] = {}\n    division_node_count = 0\n    division_evidence = {\n        key: [] for key in (\n            "source_id", "target_id", "probability",\n            "alternative_parent_probability", "is_target_winner",\n            "distance_um", "source_target_rank",\n        )\n    }\n\n    # Running node registry'), ('            del secondary_det_logits\n\n        del imgs', '            del secondary_det_logits\n\n        division_unet_out = None\n        division_det_logits = None\n        if division_model is not None:\n            division_unet_out, division_det_logits = division_model.encode(imgs)\n            # Frozen Model-C parity path: identity + x/y/xy flips.\n            division_nv = 1\n            for division_dims in [(-1,), (-2,), (-2, -1)]:\n                division_imgs_flip = imgs.flip(division_dims)\n                _, division_det_flip = division_model.encode(division_imgs_flip)\n                for division_frame in range(W):\n                    division_det_logits[division_frame] = (\n                        division_det_logits[division_frame]\n                        + division_det_flip[division_frame].flip(division_dims)\n                    )\n                del division_imgs_flip, division_det_flip\n                division_nv += 1\n            for division_frame in range(W):\n                division_det_logits[division_frame] = (\n                    division_det_logits[division_frame] / division_nv\n                )\n\n        del imgs'), ('                seen_frames.add(t)\n\n        coords_so_far = (', '                seen_frames.add(t)\n\n        if division_model is not None:\n            for division_f_idx, division_t in enumerate(frame_indices):\n                if division_t in division_seen_frames:\n                    continue\n                native = _detect_cells_pooled(\n                    division_det_logits[division_f_idx][0],\n                    division_t,\n                    division_det_threshold,\n                    division_pool_k,\n                ).astype(np.int32)\n                native[:, 1:] *= np.asarray(downsample, np.int32)\n                division_coord_by_t[division_t] = native\n                division_offset[division_t] = (\n                    division_node_count,\n                    division_node_count + len(native),\n                )\n                division_node_count += len(native)\n                division_seen_frames.add(division_t)\n\n        coords_so_far = ('), ('            seen_pairs.add((t_src, t_tgt))\n\n            if t_src not in coord_offset or t_tgt not in coord_offset:', '            seen_pairs.add((t_src, t_tgt))\n\n            if (\n                division_model is not None\n                and (t_src, t_tgt) not in division_seen_pairs\n            ):\n                division_seen_pairs.add((t_src, t_tgt))\n                native_src = division_coord_by_t.get(\n                    t_src, np.empty((0, 4), np.int32)\n                )\n                native_tgt = division_coord_by_t.get(\n                    t_tgt, np.empty((0, 4), np.int32)\n                )\n                if len(native_src) and len(native_tgt):\n                    source_raw = native_src[:, 1:].astype(np.float32)\n                    target_raw = native_tgt[:, 1:].astype(np.float32)\n                    source_ds = source_raw / ds_arr\n                    target_ds = target_raw / ds_arr\n                    n_div_source, n_div_target = len(source_ds), len(target_ds)\n                    div_src = torch.from_numpy(source_ds).unsqueeze(0).to(device)\n                    div_tgt = torch.from_numpy(target_ds).unsqueeze(0).to(device)\n                    source_rel = np.column_stack([\n                        np.full(n_div_source, f_idx, np.float32), source_ds\n                    ])\n                    target_rel = np.column_stack([\n                        np.full(n_div_target, f_idx + 1, np.float32), target_ds\n                    ])\n                    div_window_shape = (W,) + image_shape[1:]\n                    div_pos_src = torch.from_numpy(\n                        extract_pos_features(source_rel, div_window_shape)\n                    ).unsqueeze(0).to(device)\n                    div_pos_tgt = torch.from_numpy(\n                        extract_pos_features(target_rel, div_window_shape)\n                    ).unsqueeze(0).to(device)\n                    div_mask_src = torch.ones(\n                        1, n_div_source, dtype=torch.bool, device=device\n                    )\n                    div_mask_tgt = torch.ones(\n                        1, n_div_target, dtype=torch.bool, device=device\n                    )\n                    div_feat_src = division_model._index_features(\n                        division_unet_out[:, f_idx], div_src, div_mask_src\n                    )\n                    div_feat_tgt = division_model._index_features(\n                        division_unet_out[:, f_idx + 1], div_tgt, div_mask_tgt\n                    )\n                    div_logits = division_model.predict_edges(\n                        div_feat_src,\n                        div_feat_tgt,\n                        div_src * division_ds_arr_t,\n                        div_tgt * division_ds_arr_t,\n                        div_pos_src,\n                        div_pos_tgt,\n                        div_mask_src,\n                        div_mask_tgt,\n                    )[0]\n                    div_probability = torch.softmax(div_logits, dim=0).float()\n                    div_source_um = torch.as_tensor(\n                        source_raw * original_spacing, device=device\n                    ).float()\n                    div_target_um = torch.as_tensor(\n                        target_raw * original_spacing, device=device\n                    ).float()\n                    div_distance = torch.cdist(div_source_um, div_target_um)\n                    div_eligible = div_distance <= division_radius_um\n                    div_masked = div_probability.masked_fill(\n                        ~div_eligible, -1.0\n                    )\n                    div_k = min(division_topk, n_div_target)\n                    div_values, div_targets = torch.topk(\n                        div_masked, div_k, dim=1, sorted=True\n                    )\n                    div_keep = div_values >= division_min_probability\n                    if n_div_source == 1:\n                        div_best_prob = div_probability[0]\n                        div_best_row = torch.zeros_like(\n                            div_best_prob, dtype=torch.long\n                        )\n                        div_second_prob = torch.zeros_like(div_best_prob)\n                    else:\n                        div_top2 = torch.topk(\n                            div_probability, 2, dim=0, sorted=True\n                        )\n                        div_best_prob = div_top2.values[0]\n                        div_best_row = div_top2.indices[0]\n                        div_second_prob = div_top2.values[1]\n                    div_source_rows = torch.arange(\n                        n_div_source, device=device\n                    ).unsqueeze(1).expand_as(div_targets)\n                    picked_source = div_source_rows[div_keep]\n                    picked_target = div_targets[div_keep]\n                    target_best = div_best_row[picked_target]\n                    div_alternative = torch.where(\n                        target_best == picked_source,\n                        div_second_prob[picked_target],\n                        div_best_prob[picked_target],\n                    )\n                    source_start = division_offset[t_src][0]\n                    target_start = division_offset[t_tgt][0]\n                    division_evidence["source_id"].append(\n                        (picked_source + source_start).cpu().numpy().astype(\n                            np.int64\n                        )\n                    )\n                    division_evidence["target_id"].append(\n                        (picked_target + target_start).cpu().numpy().astype(\n                            np.int64\n                        )\n                    )\n                    division_evidence["probability"].append(\n                        div_values[div_keep].cpu().numpy().astype(np.float32)\n                    )\n                    division_evidence[\n                        "alternative_parent_probability"\n                    ].append(\n                        div_alternative.cpu().numpy().astype(np.float32)\n                    )\n                    division_evidence["is_target_winner"].append(\n                        (target_best == picked_source).cpu().numpy().astype(\n                            np.uint8\n                        )\n                    )\n                    division_evidence["distance_um"].append(\n                        div_distance[\n                            picked_source, picked_target\n                        ].cpu().numpy().astype(np.float32)\n                    )\n                    div_rank_grid = torch.arange(\n                        div_k, device=device\n                    ).unsqueeze(0).expand_as(div_targets)\n                    division_evidence["source_target_rank"].append(\n                        div_rank_grid[div_keep].cpu().numpy().astype(np.int16)\n                    )\n\n            if t_src not in coord_offset or t_tgt not in coord_offset:'), ('        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n', '        del unet_out\n        if secondary_unet_out is not None:\n            del secondary_unet_out\n        if division_unet_out is not None:\n            del division_unet_out\n        if division_det_logits is not None:\n            del division_det_logits\n'), ('    coords = coords.astype(np.int16)\n    return coords, all_edges\n', '    coords = coords.astype(np.int16)\n    if division_model is not None and division_evidence_path is not None:\n        native_counts = np.asarray(\n            [\n                len(division_coord_by_t.get(\n                    frame, np.empty((0, 4), np.int32)\n                ))\n                for frame in range(T)\n            ],\n            np.int32,\n        )\n        native_offsets = np.concatenate([\n            [0], np.cumsum(native_counts, dtype=np.int64)\n        ])\n        native_chunks = [\n            division_coord_by_t.get(frame, np.empty((0, 4), np.int32))\n            for frame in range(T)\n        ]\n        native_coords = (\n            np.concatenate(native_chunks).astype(np.int16)\n            if native_chunks\n            else np.empty((0, 4), np.int16)\n        )\n        evidence_dtypes = {\n            "source_id": np.int64,\n            "target_id": np.int64,\n            "probability": np.float32,\n            "alternative_parent_probability": np.float32,\n            "is_target_winner": np.uint8,\n            "distance_um": np.float32,\n            "source_target_rank": np.int16,\n        }\n        evidence_arrays = {}\n        for evidence_key, evidence_chunks in division_evidence.items():\n            evidence_arrays[evidence_key] = (\n                np.concatenate(evidence_chunks).astype(\n                    evidence_dtypes[evidence_key], copy=False\n                )\n                if evidence_chunks\n                else np.empty(0, evidence_dtypes[evidence_key])\n            )\n        division_evidence_path.parent.mkdir(parents=True, exist_ok=True)\n        evidence_tmp = division_evidence_path.with_suffix(".npz.tmp")\n        with evidence_tmp.open("wb") as evidence_handle:\n            np.savez_compressed(\n                evidence_handle,\n                **evidence_arrays,\n                native_node_coords=native_coords,\n                native_frame_offsets=native_offsets,\n                spacing_um=original_spacing.astype(np.float32),\n            )\n        evidence_tmp.replace(division_evidence_path)\n    return coords, all_edges\n'), ('    model, window_size, downsample = load_model(weights_path, device)\n\n    secondary_model = None', '    model, window_size, downsample = load_model(weights_path, device)\n\n    division_model = None\n    division_evidence_dir = None\n    division_weights_text = os.environ.get(\n        "BIOHUB_MODEL_C_CHECKPOINT", ""\n    ).strip()\n    division_evidence_text = os.environ.get(\n        "BIOHUB_MODEL_C_EVIDENCE_DIR", ""\n    ).strip()\n    division_helper_deadline_epoch = float(\n        os.environ.get("BIOHUB_HELPER_DEADLINE_EPOCH", "inf")\n    )\n    if division_weights_text:\n        division_model, division_window_size, division_downsample = load_model(\n            Path(division_weights_text), device\n        )\n        if division_window_size != window_size:\n            raise ValueError(\n                f"Model C window {division_window_size} != public {window_size}"\n            )\n        if division_downsample != downsample:\n            raise ValueError(\n                f"Model C downsample {division_downsample} != public {downsample}"\n            )\n        if not division_evidence_text:\n            raise ValueError("BIOHUB_MODEL_C_EVIDENCE_DIR is required")\n        division_evidence_dir = Path(division_evidence_text)\n        division_evidence_dir.mkdir(parents=True, exist_ok=True)\n        print(\n            f"Integrated native Model C: {division_weights_text} | "\n            f"evidence={division_evidence_dir}",\n            flush=True,\n        )\n\n    secondary_model = None'), ('    for name in tqdm(test_names, desc="Predicting", disable=not INTERACTIVE):\n        ds_path = data_dir / name\n        coords, edges = predict_video(', '    for name in tqdm(test_names, desc="Predicting", disable=not INTERACTIVE):\n        ds_path = data_dir / name\n        use_division_model = (\n            division_model is not None\n            and time.time() < division_helper_deadline_epoch\n        )\n        if division_model is not None and not use_division_model:\n            print(\n                f"Helper deadline reached; {name} keeps public baseline only",\n                flush=True,\n            )\n        coords, edges = predict_video('), ('                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n            )', '                secondary_mix_temperature=secondary_mix_temperature,\n                secondary_low_margin_max=secondary_low_margin_max,\n                division_model=(\n                    division_model if use_division_model else None\n                ),\n                division_evidence_path=(\n                    division_evidence_dir / f"{name}.npz"\n                    if use_division_model else None\n                ),\n                division_det_threshold=0.99,\n                division_pool_um=3.0,\n                division_radius_um=20.0,\n                division_topk=16,\n                division_min_probability=0.01,\n            )'), ('    output_dir.mkdir(parents=True, exist_ok=True)\n\n    device = torch.device', '    output_dir.mkdir(parents=True, exist_ok=True)\n    for stale_marker in output_dir.glob("*.ready*"):\n        stale_marker.unlink()\n\n    device = torch.device')]
_s = _apply(_s, _model_c_predictor_replacements, "Model-C evidence")

# 5. Atomic per-video ready marker, written only after save_graph() returns.
_ready_old = '        save_graph(graph, output_dir / f"{name}.geff")'
_ready_new = '''        output_path = output_dir / f"{name}.geff"
        save_graph(graph, output_path)
        ready_tmp = output_dir / f"{name}.ready.tmp"
        ready_tmp.write_text("complete\\n")
        ready_tmp.replace(output_dir / f"{name}.ready")'''
_s = _apply(_s, [(_ready_old, _ready_new)], "ready marker")

# 6. In-pass P1/P2 source-cardinality evidence (no second GPU sweep).
_inpass_native_evidence_replacements = [('@torch.no_grad()\ndef predict_video(\n', '\n\ndef _new_native_evidence_state():\n    return {\n        "coords_by_t": {},\n        "offsets": {},\n        "node_count": 0,\n        "seen_frames": set(),\n        "seen_pairs": set(),\n        "evidence": {\n            key: [] for key in (\n                "source_id", "target_id", "probability",\n                "alternative_parent_probability", "is_target_winner",\n                "distance_um", "source_target_rank",\n            )\n        },\n    }\n\n\ndef _detect_native_evidence_frames(\n    state,\n    detection,\n    frame_indices,\n    threshold,\n    pool_kernel,\n    downsample,\n):\n    scale = np.asarray(downsample, np.int32)\n    for local_frame, absolute_frame in enumerate(frame_indices):\n        if absolute_frame in state["seen_frames"]:\n            continue\n        coord = _detect_cells_pooled(\n            detection[local_frame][0],\n            absolute_frame,\n            threshold,\n            pool_kernel,\n        ).astype(np.int32)\n        coord[:, 1:] *= scale\n        start = int(state["node_count"])\n        state["coords_by_t"][absolute_frame] = coord\n        state["offsets"][absolute_frame] = (start, start + len(coord))\n        state["node_count"] = start + len(coord)\n        state["seen_frames"].add(absolute_frame)\n\n\ndef _append_native_evidence_pair(\n    state,\n    model,\n    feature_maps,\n    f_idx,\n    t_src,\n    t_tgt,\n    downsample,\n    spacing,\n    window_shape,\n    device,\n    radius_um,\n    topk_per_source,\n    min_probability,\n):\n    if (t_src, t_tgt) in state["seen_pairs"]:\n        return\n    state["seen_pairs"].add((t_src, t_tgt))\n    source_native = state["coords_by_t"].get(t_src)\n    target_native = state["coords_by_t"].get(t_tgt)\n    if source_native is None or target_native is None:\n        return\n    if not len(source_native) or not len(target_native):\n        return\n    downsample_array = np.asarray(downsample, np.float32)\n    source_raw = source_native[:, 1:].astype(np.float32)\n    target_raw = target_native[:, 1:].astype(np.float32)\n    source_downsampled = source_raw / downsample_array\n    target_downsampled = target_raw / downsample_array\n    n_source, n_target = len(source_downsampled), len(target_downsampled)\n    source_tensor = torch.from_numpy(source_downsampled).unsqueeze(0).to(device)\n    target_tensor = torch.from_numpy(target_downsampled).unsqueeze(0).to(device)\n    source_relative = np.column_stack([\n        np.full(n_source, f_idx, np.float32), source_downsampled,\n    ])\n    target_relative = np.column_stack([\n        np.full(n_target, f_idx + 1, np.float32), target_downsampled,\n    ])\n    source_position = torch.from_numpy(\n        extract_pos_features(source_relative, window_shape).astype(np.float32)\n    ).unsqueeze(0).to(device)\n    target_position = torch.from_numpy(\n        extract_pos_features(target_relative, window_shape).astype(np.float32)\n    ).unsqueeze(0).to(device)\n    source_mask = torch.ones(1, n_source, dtype=torch.bool, device=device)\n    target_mask = torch.ones(1, n_target, dtype=torch.bool, device=device)\n    source_features = model._index_features(\n        feature_maps[:, f_idx], source_tensor, source_mask,\n    )\n    target_features = model._index_features(\n        feature_maps[:, f_idx + 1], target_tensor, target_mask,\n    )\n    downsample_tensor = torch.as_tensor(\n        downsample_array, dtype=torch.float32, device=device,\n    )\n    logits = model.predict_edges(\n        source_features,\n        target_features,\n        source_tensor * downsample_tensor,\n        target_tensor * downsample_tensor,\n        source_position,\n        target_position,\n        source_mask,\n        target_mask,\n    )[0]\n    probability = torch.softmax(logits, dim=0).float()\n    spacing_array = np.asarray(spacing, np.float32)\n    source_um = torch.as_tensor(\n        source_raw * spacing_array, dtype=torch.float32, device=device,\n    )\n    target_um = torch.as_tensor(\n        target_raw * spacing_array, dtype=torch.float32, device=device,\n    )\n    distance = torch.cdist(source_um, target_um)\n    eligible = distance <= radius_um\n    masked = probability.masked_fill(~eligible, -1.0)\n    k = min(topk_per_source, n_target)\n    values, targets = torch.topk(masked, k, dim=1, sorted=True)\n    keep = values >= min_probability\n    if n_source == 1:\n        best_probability = probability[0]\n        best_parent = torch.zeros_like(best_probability, dtype=torch.long)\n        second_probability = torch.zeros_like(best_probability)\n    else:\n        top2 = torch.topk(probability, 2, dim=0, sorted=True)\n        best_probability = top2.values[0]\n        best_parent = top2.indices[0]\n        second_probability = top2.values[1]\n    source_rows = torch.arange(\n        n_source, device=device,\n    ).unsqueeze(1).expand_as(targets)\n    picked_source = source_rows[keep]\n    picked_target = targets[keep]\n    target_best_parent = best_parent[picked_target]\n    alternative = torch.where(\n        target_best_parent == picked_source,\n        second_probability[picked_target],\n        best_probability[picked_target],\n    )\n    source_start = int(state["offsets"][t_src][0])\n    target_start = int(state["offsets"][t_tgt][0])\n    evidence = state["evidence"]\n    evidence["source_id"].append(\n        (picked_source + source_start).cpu().numpy().astype(np.int64)\n    )\n    evidence["target_id"].append(\n        (picked_target + target_start).cpu().numpy().astype(np.int64)\n    )\n    evidence["probability"].append(\n        values[keep].cpu().numpy().astype(np.float32)\n    )\n    evidence["alternative_parent_probability"].append(\n        alternative.cpu().numpy().astype(np.float32)\n    )\n    evidence["is_target_winner"].append(\n        (target_best_parent == picked_source).cpu().numpy().astype(np.uint8)\n    )\n    evidence["distance_um"].append(\n        distance[picked_source, picked_target].cpu().numpy().astype(np.float32)\n    )\n    rank_grid = torch.arange(\n        k, device=device,\n    ).unsqueeze(0).expand_as(targets)\n    evidence["source_target_rank"].append(\n        rank_grid[keep].cpu().numpy().astype(np.int16)\n    )\n\n\ndef _save_native_evidence(state, output_path, frames, spacing):\n    if output_path is None:\n        return\n    empty_coords = np.empty((0, 4), np.int32)\n    chunks = [\n        state["coords_by_t"].get(frame, empty_coords)\n        for frame in range(frames)\n    ]\n    counts = np.asarray([len(chunk) for chunk in chunks], np.int32)\n    offsets = np.concatenate([[0], np.cumsum(counts, dtype=np.int64)])\n    native_coords = (\n        np.concatenate(chunks).astype(np.int16, copy=False)\n        if chunks else np.empty((0, 4), np.int16)\n    )\n    dtypes = {\n        "source_id": np.int64,\n        "target_id": np.int64,\n        "probability": np.float32,\n        "alternative_parent_probability": np.float32,\n        "is_target_winner": np.uint8,\n        "distance_um": np.float32,\n        "source_target_rank": np.int16,\n    }\n    arrays = {}\n    for key, evidence_chunks in state["evidence"].items():\n        arrays[key] = (\n            np.concatenate(evidence_chunks).astype(dtypes[key], copy=False)\n            if evidence_chunks else np.empty(0, dtypes[key])\n        )\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output_path.with_suffix(".npz.tmp")\n    with temporary.open("wb") as handle:\n        np.savez_compressed(\n            handle,\n            **arrays,\n            native_node_coords=native_coords,\n            native_frame_offsets=offsets,\n            spacing_um=np.asarray(spacing, np.float32),\n        )\n    temporary.replace(output_path)\n\n\n@torch.no_grad()\ndef predict_video(\n'), ('    division_min_probability: float = 0.01,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:', '    division_min_probability: float = 0.01,\n    primary_native_evidence_path: Path | None = None,\n    secondary_native_evidence_path: Path | None = None,\n    native_evidence_det_threshold: float = 0.96875,\n    native_evidence_pool_um: float = 5.0,\n    native_evidence_radius_um: float = 20.0,\n    native_evidence_topk: int = 16,\n    native_evidence_min_probability: float = 0.01,\n) -> tuple[np.ndarray, list[tuple[int, int, float, float]]]:'), ('    division_evidence = {\n        key: [] for key in (\n            "source_id", "target_id", "probability",\n            "alternative_parent_probability", "is_target_winner",\n            "distance_um", "source_target_rank",\n        )\n    }\n\n    # Running node registry', '    division_evidence = {\n        key: [] for key in (\n            "source_id", "target_id", "probability",\n            "alternative_parent_probability", "is_target_winner",\n            "distance_um", "source_target_rank",\n        )\n    }\n    native_evidence_pool_kernel = pool_kernel_from_um(\n        native_evidence_pool_um, voxel_size,\n    )\n    primary_native_state = (\n        _new_native_evidence_state()\n        if primary_native_evidence_path is not None else None\n    )\n    secondary_native_state = (\n        _new_native_evidence_state()\n        if secondary_native_evidence_path is not None else None\n    )\n\n    # Running node registry'), ('        if cfg.det_tta:\n            _nv = 1', '        primary_native_detection = None\n        if cfg.det_tta:\n            _nv = 1'), ('                del imgs_flip, det_flip\n                _nv += 1\n            for _k in (1, 3):', '                del imgs_flip, det_flip\n                _nv += 1\n            primary_native_detection = [value / _nv for value in det_logits]\n            for _k in (1, 3):'), ('            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n\n        secondary_unet_out = None', '            for f in range(W):\n                det_logits[f] = det_logits[f] / _nv\n        if primary_native_detection is None:\n            primary_native_detection = [value.clone() for value in det_logits]\n\n        secondary_unet_out = None'), ('        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:', '        secondary_native_detection = None\n        if secondary_model is not None:\n            secondary_unet_out, secondary_det_logits = secondary_model.encode(imgs)\n\n            if secondary_detection_weight > 0.0:'), ('                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    for _k in (1, 3):', '                        del secondary_imgs_flip, secondary_det_flip\n                        _secondary_nv += 1\n                    secondary_native_detection = [\n                        value / _secondary_nv for value in secondary_det_logits\n                    ]\n                    for _k in (1, 3):'), ('                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n\n                for f in range(W):', '                    for f in range(W):\n                        secondary_det_logits[f] = secondary_det_logits[f] / _secondary_nv\n                if secondary_native_detection is None:\n                    secondary_native_detection = [\n                        value.clone() for value in secondary_det_logits\n                    ]\n\n                for f in range(W):'), ('        del imgs\n\n        # --- Detect cells in each frame (dedup across windows) ---', '        del imgs\n\n        if primary_native_state is not None:\n            _detect_native_evidence_frames(\n                primary_native_state,\n                primary_native_detection,\n                frame_indices,\n                native_evidence_det_threshold,\n                native_evidence_pool_kernel,\n                downsample,\n            )\n        if secondary_native_state is not None:\n            if secondary_native_detection is None:\n                raise RuntimeError("Secondary native evidence requested without model")\n            _detect_native_evidence_frames(\n                secondary_native_state,\n                secondary_native_detection,\n                frame_indices,\n                native_evidence_det_threshold,\n                native_evidence_pool_kernel,\n                downsample,\n            )\n\n        # --- Detect cells in each frame (dedup across windows) ---'), ('            seen_pairs.add((t_src, t_tgt))\n\n            if (\n                division_model is not None', '            seen_pairs.add((t_src, t_tgt))\n\n            native_window_shape = (W,) + image_shape[1:]\n            if primary_native_state is not None:\n                _append_native_evidence_pair(\n                    primary_native_state, model, unet_out,\n                    f_idx, t_src, t_tgt, downsample, ds.scale,\n                    native_window_shape, device,\n                    native_evidence_radius_um,\n                    native_evidence_topk,\n                    native_evidence_min_probability,\n                )\n            if secondary_native_state is not None:\n                if secondary_model is None or secondary_unet_out is None:\n                    raise RuntimeError("Secondary native evidence feature map missing")\n                _append_native_evidence_pair(\n                    secondary_native_state, secondary_model, secondary_unet_out,\n                    f_idx, t_src, t_tgt, downsample, ds.scale,\n                    native_window_shape, device,\n                    native_evidence_radius_um,\n                    native_evidence_topk,\n                    native_evidence_min_probability,\n                )\n\n            if (\n                division_model is not None'), ('        if division_det_logits is not None:\n            del division_det_logits\n\n    coords = np.concatenate(coord_lists)', '        if division_det_logits is not None:\n            del division_det_logits\n        if primary_native_detection is not None:\n            del primary_native_detection\n        if secondary_native_detection is not None:\n            del secondary_native_detection\n\n    coords = np.concatenate(coord_lists)'), ('        evidence_tmp.replace(division_evidence_path)\n    return coords, all_edges', '        evidence_tmp.replace(division_evidence_path)\n    _save_native_evidence(\n        primary_native_state,\n        primary_native_evidence_path,\n        T,\n        ds.scale,\n    ) if primary_native_state is not None else None\n    _save_native_evidence(\n        secondary_native_state,\n        secondary_native_evidence_path,\n        T,\n        ds.scale,\n    ) if secondary_native_state is not None else None\n    return coords, all_edges'), ('    division_model = None\n    division_evidence_dir = None', '    primary_native_evidence_dir = None\n    secondary_native_evidence_dir = None\n    primary_native_evidence_text = os.environ.get(\n        "BIOHUB_PUBLIC_P1_EVIDENCE_DIR", ""\n    ).strip()\n    secondary_native_evidence_text = os.environ.get(\n        "BIOHUB_PUBLIC_P2_EVIDENCE_DIR", ""\n    ).strip()\n    if primary_native_evidence_text:\n        primary_native_evidence_dir = Path(primary_native_evidence_text)\n        primary_native_evidence_dir.mkdir(parents=True, exist_ok=True)\n    if secondary_native_evidence_text:\n        secondary_native_evidence_dir = Path(secondary_native_evidence_text)\n        secondary_native_evidence_dir.mkdir(parents=True, exist_ok=True)\n\n    division_model = None\n    division_evidence_dir = None'), ('                division_min_probability=0.01,\n            )', '                division_min_probability=0.01,\n                primary_native_evidence_path=(\n                    primary_native_evidence_dir / f"{name}.npz"\n                    if use_division_model and primary_native_evidence_dir is not None\n                    else None\n                ),\n                secondary_native_evidence_path=(\n                    secondary_native_evidence_dir / f"{name}.npz"\n                    if use_division_model and secondary_native_evidence_dir is not None\n                    else None\n                ),\n                native_evidence_det_threshold=0.96875,\n                native_evidence_pool_um=5.0,\n                native_evidence_radius_um=20.0,\n                native_evidence_topk=16,\n                native_evidence_min_probability=0.01,\n            )')]
_s = _apply(_s, _inpass_native_evidence_replacements, "in-pass P1/P2 evidence")

_ps.write_text(_s)
print("predict_unet_transformer.py: 6 patch groups applied")

In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, f"{PKG_DIR}/patch_predictor.py"], check=True, env=dict(os.environ)
)

In [ ]:
%%writefile /kaggle/working/biohub_pipeline/postprocess.py
"""Output-graph post-processing for the Biohub cell-tracking submission.

The notebook imports this module once in the parent process; the forked CPU
workers inherit the already-loaded runtimes. Every numeric constant comes from
the CFG JSON emitted by the notebook's CFG cell, so this file has no tuning of
its own.

Pipeline per video graph:
    raw ILP edges -> motion re-link -> single-parent repair -> gap closing
    -> divisions (combined Model-C/V2 decoder, or the safe-geometry fallback)
    -> isolated pruning -> short-track filtering -> line-fit smoothing
"""
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist

CFG = json.loads(Path(os.environ["BIOHUB_CFG_JSON"]).read_text())
_PATHS = CFG["paths"]
_G = CFG["graph"]
_D = CFG["division"]
_M = CFG["models"]

TEST_DIR = Path(_PATHS["test_dir"])
SHARD_DIR = Path(_PATHS["shard_dir"])
MODEL_C_EVIDENCE_DIR = Path(_PATHS["model_c_evidence_dir"])
P1_EVIDENCE_DIR = Path(_PATHS["p1_evidence_dir"])
P2_EVIDENCE_DIR = Path(_PATHS["p2_evidence_dir"])

VOXEL_SCALE_UM = tuple(float(value) for value in CFG["voxel_scale_um"])
SCALE = np.asarray(VOXEL_SCALE_UM, dtype=np.float64)

SUBMISSION_COLUMNS = [
    "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id",
]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]

EDGE_MAX_UM = float(_G["edge_max_um"])
ENFORCE_NEXT_FRAME = bool(_G["enforce_next_frame"])
SINGLE_PARENT_REPAIR = bool(_G["single_parent_repair"])
PRUNE_ISOLATED = bool(_G["prune_isolated"])

MOTION_RELINK = bool(_G["motion_relink"])
RELINK_TIGHT_UM = float(_G["relink_tight_um"])
RELINK_RELAXED_UM = float(_G["relink_relaxed_um"])
RELINK_VELOCITY_WEIGHT = float(_G["relink_velocity_weight"])
RELINK_LEARNED_BONUS = float(_G["relink_learned_bonus"])
RELINK_MAX_FRAME_NODES = int(_G["relink_max_frame_nodes"])
RELINK_RAW_WEIGHT = float(_G["relink_raw_distance_weight"])
Z_EXTENT_UM = float(_G["motion_feature_z_extent_um"])

GAP_CLOSE = bool(_G["gap_close"])
GAP_CLOSE_UM = float(_G["gap_close_um"])
GAP_DENSITY_ADAPTIVE = bool(_G["gap_density_adaptive"])
GAP_DENSITY_REFERENCE_UM = float(_G["gap_density_reference_um"])
GAP_DENSITY_GAIN = float(_G["gap_density_gain"])
GAP_DENSITY_MAX_STEP_DELTA_UM = float(_G["gap_density_max_step_delta_um"])
GAP_DENSITY_NEIGHBORS = int(_G["gap_density_neighbors"])
GAP_REUSE_EXISTING = bool(_G["gap_reuse_existing"])
GAP_REUSE_UM = float(_G["gap_reuse_um"])
GAP_MAX_ADDED_FRAC = float(_G["gap_max_added_frac"])
GAP_MAX_ADDED_ABS = int(_G["gap_max_added_abs"])
GAP_REFINE_SYNTHETIC = bool(_G["gap_refine_synthetic"])
GAP_REFINE_WIN_Z = int(_G["gap_refine_win_z"])
GAP_REFINE_WIN_YX = int(_G["gap_refine_win_yx"])
GAP_REFINE_MAX_SHIFT_UM = float(_G["gap_refine_max_shift_um"])

SAFE_DIVISIONS = bool(_G["safe_divisions"])
SAFE_DIV_MAX_UM = float(_G["safe_div_max_um"])
SAFE_DIV_SISTER_MAX_UM = float(_G["safe_div_sister_max_um"])
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(_G["safe_div_existing_child_max_um"])
SAFE_DIV_FRAME_FRAC_CAP = float(_G["safe_div_frame_frac_cap"])
SAFE_DIV_GLOBAL_FRAC_CAP = float(_G["safe_div_global_frac_cap"])
SAFE_DIV_SISTER_SCORE_WEIGHT = float(_G["safe_div_sister_score_weight"])

FILTER_SHORT_TRACKS = bool(_G["filter_short_tracks"])
MIN_TRACK_LEN = int(_G["min_track_len"])
KEEP_DIVISION_COMPONENTS = bool(_G["keep_division_components"])

LINEFIT_SMOOTH = bool(_G["linefit_smooth"])
LINEFIT_WEIGHT = float(_G["linefit_weight"])
LINEFIT_WINDOW = int(_G["linefit_window"])
FRAME_CACHE_MAX = int(_G["frame_cache_max_frames"])

COMBINED_DECODER = bool(_D["combined_decoder"])
MODEL_C_DIR = Path(_M["model_c"]["dir"])
MODEL_C_THRESHOLD = float(_D["model_c_threshold"])
GBM_THRESHOLD = float(_D["gbm_threshold"])
GBM_RESCUE_DELTA = float(_D["gbm_rescue_delta"])
GBM_STEAL_DELTA = float(_D["gbm_steal_delta"])
SOURCE_CARDINALITY_DIR = Path(_M["source_cardinality"]["dir"])
SOURCE_CARDINALITY_ENABLED = bool(_M["source_cardinality"]["enabled"])

MOTION_CORRECTOR_ENABLED = bool(_M["motion_corrector"]["enabled"])
MOTION_CORRECTOR_PATH = Path(_M["motion_corrector"]["path"])
MOTION_CORRECTOR_STRENGTH = float(_M["motion_corrector"]["strength"])

BASE_STATS = {
    "raw_edges": 0,
    "dropped_nonconsecutive_edges": 0,
    "dropped_long_edges": 0,
    "dropped_multi_parent_edges": 0,
    "motion_relink_edges": 0,
    "motion_relink_tight_edges": 0,
    "motion_relink_relaxed_edges": 0,
    "motion_relink_frames": 0,
    "motion_relink_replaced_raw_edges": 0,
    "motion_relink_fallback_raw": 0,
    "motion_relink_skipped_large_frame": 0,
    "motion_corrector_candidates": 0,
    "motion_corrector_abs_residual_sum": 0.0,
    "gap_candidates": 0,
    "gap_pairs_selected": 0,
    "gap_reused_existing": 0,
    "gap_inserted_synthetic": 0,
    "gap_added_nodes": 0,
    "gap_added_edges": 0,
    "gap_skipped_node_cap": 0,
    "gap_density_candidates_expanded": 0,
    "gap_density_candidates_restricted": 0,
    "gap_density_selected_outside_base": 0,
    "gap_refined_synthetic": 0,
    "gap_refine_failed": 0,
    "gap_refine_rejected_shift": 0,
    "safe_division_candidates": 0,
    "safe_divisions_added": 0,
    "safe_division_skipped_cap": 0,
    "pruned_isolated_nodes": 0,
    "short_track_components_removed": 0,
    "short_track_nodes_removed": 0,
    "short_track_edges_removed": 0,
    "linefit_smoothed_nodes": 0,
    "linefit_skipped_nodes": 0,
}


# ---------------------------------------------------------------------------
# Geometry helpers
# ---------------------------------------------------------------------------
def node_point(node: dict) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def point_distance_um(a, b) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def edge_distance_um(source: dict, target: dict) -> float:
    return point_distance_um(node_point(source), node_point(target))


def edge_sort_key(edge: dict) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    return (float(prob) if prob is not None else 0.0, -float(edge["distance_um"]))


def _coords_um(nodes_by_id: dict, ids) -> np.ndarray:
    """(n, 3) micrometre positions for `ids`, in the given order."""
    out = np.empty((len(ids), 3), dtype=np.float64)
    for index, node_id in enumerate(ids):
        node = nodes_by_id[node_id]
        out[index, 0] = node["z"]
        out[index, 1] = node["y"]
        out[index, 2] = node["x"]
    return out * SCALE


def _next_node_id(nodes_by_id: dict) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1


def _ids_by_frame(nodes_by_id: dict) -> dict[int, list[int]]:
    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()
    return ids_by_t


# ---------------------------------------------------------------------------
# Raw volume access (only needed to refine synthetic gap nodes)
# ---------------------------------------------------------------------------
def read_test_frame(dataset: str, t: int, frame_cache: dict) -> np.ndarray:
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    frame_shape = tuple(int(value) for value in meta["shape"])[1:]
    dtype = np.dtype(meta["data_type"])
    frame = None
    try:
        import blosc2

        raw = (zarr_path / "0" / "c" / str(t) / "0" / "0" / "0").read_bytes()
        flat = np.frombuffer(blosc2.decompress(raw), dtype=dtype)
        if flat.size == int(np.prod(frame_shape)):
            frame = flat.reshape(frame_shape).copy()
    except Exception:
        frame = None
    if frame is None:
        import zarr

        frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])
    frame_cache[t] = frame
    while len(frame_cache) > FRAME_CACHE_MAX:
        frame_cache.pop(next(iter(frame_cache)))
    return frame


def refine_synthetic_midpoint(dataset, t, midpoint, frame_cache, stats):
    """Snap an interpolated gap node onto the local intensity centroid."""
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = (int(round(value)) for value in midpoint)
        z0, z1 = max(0, z - GAP_REFINE_WIN_Z), min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
        y0, y1 = max(0, y - GAP_REFINE_WIN_YX), min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
        x0, x1 = max(0, x - GAP_REFINE_WIN_YX), min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        weights = np.maximum(patch - float(np.percentile(patch, 20.0)), 0.0)
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        refined = (
            float((weights.sum(axis=(1, 2)) * np.arange(z0, z1)).sum() / total),
            float((weights.sum(axis=(0, 2)) * np.arange(y0, y1)).sum() / total),
            float((weights.sum(axis=(0, 1)) * np.arange(x0, x1)).sum() / total),
        )
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1
            return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined
    except Exception:
        stats["gap_refine_failed"] += 1
        return midpoint


# ---------------------------------------------------------------------------
# Learned motion-cost corrector
# ---------------------------------------------------------------------------
MOTION_FEATURES = [
    "base_cost", "raw_dist", "registered_dist", "motion_dist",
    "abs_dz", "abs_dy", "abs_dx", "abs_reg_dz", "abs_reg_dy", "abs_reg_dx",
    "velocity_z", "velocity_y", "velocity_x", "velocity_mag",
    "shift_z", "shift_y", "shift_x", "shift_mag",
    "density_src", "density_tgt", "z_boundary_src", "z_boundary_tgt",
    "has_predecessor",
]


class _MotionResidual(torch.nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(n_features, 64), torch.nn.SiLU(), torch.nn.Dropout(0.05),
            torch.nn.Linear(64, 32), torch.nn.SiLU(), torch.nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _load_motion_corrector():
    if not MOTION_CORRECTOR_ENABLED:
        return None
    checkpoint = torch.load(MOTION_CORRECTOR_PATH, map_location="cpu", weights_only=False)
    if list(checkpoint["features"]) != MOTION_FEATURES:
        raise ValueError(f"Unexpected motion-corrector features: {checkpoint['features']}")
    model = _MotionResidual(len(MOTION_FEATURES))
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return {
        "model": model,
        "mean": torch.as_tensor(checkpoint["mean"], dtype=torch.float32),
        "std": torch.as_tensor(checkpoint["std"], dtype=torch.float32),
        "residual_scale": float(checkpoint["residual_scale"]),
    }


MOTION_CORRECTOR = _load_motion_corrector()


@torch.no_grad()
def _motion_cost_residual(features: np.ndarray) -> np.ndarray:
    if MOTION_CORRECTOR is None or features.size == 0:
        return np.zeros(len(features), dtype=np.float64)
    x = torch.from_numpy(np.ascontiguousarray(features, dtype=np.float32))
    x = (x - MOTION_CORRECTOR["mean"]) / MOTION_CORRECTOR["std"]
    scale = MOTION_CORRECTOR["residual_scale"]
    raw = MOTION_CORRECTOR["model"](x)
    return (scale * torch.tanh(raw / scale)).numpy().astype(np.float64)


def _sanitized_edge_probs(edges: list[dict]) -> dict[tuple[int, int], float]:
    """Best finite probability per (source, target), squashed into [0, 1]."""
    out: dict[tuple[int, int], float] = {}
    for edge in edges:
        prob = edge.get("edge_prob")
        if prob is None:
            continue
        try:
            value = float(prob)
        except (TypeError, ValueError):
            continue
        if not np.isfinite(value):
            continue
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        key = (int(edge["source_id"]), int(edge["target_id"]))
        if value > out.get(key, -1.0):
            out[key] = value
    return out


# ---------------------------------------------------------------------------
# Motion-aware re-linking
# ---------------------------------------------------------------------------
def motion_relink_edges(nodes_by_id: dict, stats, learned_edge_probs=None) -> list[dict]:
    """Rebuild the frame-to-frame linking by motion-predicted bipartite matching.

    Cost for a candidate pair is ``|q_t - q_pred| + w_raw * |q_t - q_s|`` minus
    the learned association bonus and the learned motion residual. Candidates
    are gated by a KD-tree, so the assignment matrix is the only dense object.
    """
    if not MOTION_RELINK or not nodes_by_id:
        return []

    ids_by_t = _ids_by_frame(nodes_by_id)
    if max((len(ids) for ids in ids_by_t.values()), default=0) > RELINK_MAX_FRAME_NODES:
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    probs = learned_edge_probs or {}
    position: dict[int, np.ndarray] = {}
    density: dict[int, np.ndarray] = {}
    for t, ids in ids_by_t.items():
        coords = _coords_um(nodes_by_id, ids)
        position[t] = coords
        counts = cKDTree(coords).query_ball_point(coords, 15.0, return_length=True)
        density[t] = np.maximum(counts.astype(np.float64) - 1.0, 0.0)

    previous_position: dict[int, np.ndarray] = {}
    selected: list[dict] = []

    for t in sorted(ids_by_t):
        source_ids = ids_by_t.get(t, [])
        target_ids = ids_by_t.get(t + 1, [])
        if not source_ids or not target_ids:
            continue

        source_pos = position[t]
        target_pos = position[t + 1]
        velocity = np.zeros_like(source_pos)
        has_predecessor = np.zeros(len(source_ids), dtype=np.float64)
        for index, node_id in enumerate(source_ids):
            previous = previous_position.get(node_id)
            if previous is not None:
                velocity[index] = source_pos[index] - previous
                has_predecessor[index] = 1.0
        predicted = source_pos + RELINK_VELOCITY_WEIGHT * velocity
        velocity_magnitude = np.linalg.norm(velocity, axis=1)
        z_boundary_src = np.clip(source_pos[:, 0] / Z_EXTENT_UM, 0.0, 1.0)
        z_boundary_src = np.minimum(z_boundary_src, 1.0 - z_boundary_src)
        z_boundary_tgt = np.clip(target_pos[:, 0] / Z_EXTENT_UM, 0.0, 1.0)
        z_boundary_tgt = np.minimum(z_boundary_tgt, 1.0 - z_boundary_tgt)

        def assign_pass(source_sel: np.ndarray, target_sel: np.ndarray, gate_um: float):
            if source_sel.size == 0 or target_sel.size == 0:
                return np.empty(0, np.int64), np.empty(0, np.int64)
            local_source = source_pos[source_sel]
            local_target = target_pos[target_sel]
            neighbours = cKDTree(local_target).query_ball_point(local_source, gate_um)
            counts = np.fromiter((len(item) for item in neighbours), np.int64, len(neighbours))
            if not counts.any():
                return np.empty(0, np.int64), np.empty(0, np.int64)
            rows = np.repeat(np.arange(len(source_sel), dtype=np.int64), counts)
            cols = np.fromiter(
                (index for item in neighbours for index in item), np.int64, int(counts.sum())
            )
            delta = local_target[cols] - local_source[rows]
            raw = np.linalg.norm(delta, axis=1)
            inside = raw <= gate_um
            rows, cols, delta, raw = rows[inside], cols[inside], delta[inside], raw[inside]
            if rows.size == 0:
                return np.empty(0, np.int64), np.empty(0, np.int64)

            global_source = source_sel[rows]
            global_target = target_sel[cols]
            motion = np.linalg.norm(
                local_target[cols] - predicted[global_source], axis=1
            )
            probability = np.fromiter(
                (
                    probs.get((source_ids[s], target_ids[g]), 0.0)
                    for s, g in zip(global_source, global_target)
                ),
                np.float64,
                rows.size,
            )
            base_cost = motion + RELINK_RAW_WEIGHT * raw
            values = base_cost - RELINK_LEARNED_BONUS * probability

            if MOTION_CORRECTOR is not None:
                absolute = np.abs(delta)
                features = np.column_stack([
                    base_cost, raw, raw, motion,
                    absolute, absolute,
                    velocity[global_source], velocity_magnitude[global_source],
                    np.zeros((rows.size, 4)),
                    density[t][global_source], density[t + 1][global_target],
                    z_boundary_src[global_source], z_boundary_tgt[global_target],
                    has_predecessor[global_source],
                ])
                residual = _motion_cost_residual(features) * MOTION_CORRECTOR_STRENGTH
                values = values - residual
                stats["motion_corrector_candidates"] += int(rows.size)
                stats["motion_corrector_abs_residual_sum"] += float(np.abs(residual).sum())

            big = gate_um * 1000.0 + 1.0
            cost = np.full((source_sel.size, target_sel.size), big, dtype=np.float64)
            cost[rows, cols] = values
            row_index, col_index = linear_sum_assignment(cost)
            matched = cost[row_index, col_index] < big
            return source_sel[row_index[matched]], target_sel[col_index[matched]]

        open_sources = np.ones(len(source_ids), dtype=bool)
        open_targets = np.ones(len(target_ids), dtype=bool)
        for pass_name, gate_um in (("tight", RELINK_TIGHT_UM), ("relaxed", RELINK_RELAXED_UM)):
            matched_source, matched_target = assign_pass(
                np.flatnonzero(open_sources), np.flatnonzero(open_targets), gate_um
            )
            if matched_source.size == 0:
                continue
            open_sources[matched_source] = False
            open_targets[matched_target] = False
            stats[f"motion_relink_{pass_name}_edges"] += int(matched_source.size)

            raw = np.linalg.norm(target_pos[matched_target] - source_pos[matched_source], axis=1)
            motion = np.linalg.norm(
                target_pos[matched_target] - predicted[matched_source], axis=1
            )
            for offset in range(matched_source.size):
                source_id = source_ids[int(matched_source[offset])]
                target_id = target_ids[int(matched_target[offset])]
                selected.append({
                    "source_id": source_id,
                    "target_id": target_id,
                    "edge_prob": probs.get((source_id, target_id), 0.0),
                    "distance_um": float(raw[offset]),
                    "motion_distance_um": float(motion[offset]),
                    "motion_relinked": 1,
                    "motion_pass": pass_name,
                })
                previous_position[target_id] = source_pos[int(matched_source[offset])]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected)
    return selected


# ---------------------------------------------------------------------------
# Single-frame gap closing
# ---------------------------------------------------------------------------
def close_single_frame_gaps(nodes_by_id, edges, stats, dataset=None, frame_cache=None):
    """Bridge one-frame track interruptions with a reused or synthetic node."""
    if not GAP_CLOSE or not edges:
        return nodes_by_id, edges

    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    all_ids_by_t = _ids_by_frame(nodes_by_id)
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
        if node_id not in incident:
            isolated_by_t.setdefault(t, []).append(node_id)
    for bucket in (ends_by_t, starts_by_t, isolated_by_t):
        for ids in bucket.values():
            ids.sort()

    max_synthetic = min(
        GAP_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_MAX_ADDED_FRAC))) if GAP_MAX_ADDED_FRAC > 0 else 0,
    )
    next_id = _next_node_id(nodes_by_id)
    frame_cache = {} if frame_cache is None else frame_cache
    used_starts: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict] = []

    spacing_cache: dict[int, np.ndarray] = {}

    def frame_spacing(t: int) -> np.ndarray:
        """Median distance to the nearest neighbours, per node of frame `t`."""
        cached = spacing_cache.get(t)
        if cached is not None:
            return cached
        ids = all_ids_by_t.get(t, [])
        if len(ids) <= 1:
            result = np.full(len(ids), GAP_DENSITY_REFERENCE_UM)
        else:
            coords = _coords_um(nodes_by_id, ids)
            k = min(len(ids), max(2, GAP_DENSITY_NEIGHBORS + 1))
            distances = np.atleast_2d(cKDTree(coords).query(coords, k=k)[0])[:, 1:]
            distances = np.where(np.isfinite(distances), distances, np.nan)
            with np.errstate(invalid="ignore"):
                result = np.nanmedian(distances, axis=1)
            result = np.where(np.isfinite(result), result, GAP_DENSITY_REFERENCE_UM)
        spacing_cache[t] = result
        return result

    isolated_cache: dict[int, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}

    def isolated_pool(t: int):
        cached = isolated_cache.get(t)
        if cached is None:
            ids = np.asarray(isolated_by_t.get(t, []), dtype=np.int64)
            cached = (ids, _coords_um(nodes_by_id, ids), np.ones(ids.size, dtype=bool))
            isolated_cache[t] = cached
        return cached

    threshold_um = GAP_CLOSE_UM * 2.0
    for t, end_ids in sorted(ends_by_t.items()):
        start_ids = [node_id for node_id in starts_by_t.get(t + 2, []) if node_id not in used_starts]
        if not end_ids or not start_ids:
            continue

        distance = cdist(
            _coords_um(nodes_by_id, end_ids), _coords_um(nodes_by_id, start_ids)
        )
        adaptive = np.full_like(distance, threshold_um)
        if GAP_DENSITY_ADAPTIVE:
            end_index = {node_id: i for i, node_id in enumerate(all_ids_by_t[t])}
            start_index = {node_id: i for i, node_id in enumerate(all_ids_by_t[t + 2])}
            source_spacing = frame_spacing(t)[[end_index[i] for i in end_ids]]
            target_spacing = frame_spacing(t + 2)[[start_index[j] for j in start_ids]]
            local = 0.5 * (source_spacing[:, None] + target_spacing[None, :])
            step = np.clip(
                GAP_DENSITY_GAIN * (local - GAP_DENSITY_REFERENCE_UM),
                -GAP_DENSITY_MAX_STEP_DELTA_UM,
                GAP_DENSITY_MAX_STEP_DELTA_UM,
            )
            adaptive = threshold_um + 2.0 * step

        base_allowed = distance <= threshold_um
        allowed = distance <= adaptive
        stats["gap_density_candidates_expanded"] += int((allowed & ~base_allowed).sum())
        stats["gap_density_candidates_restricted"] += int((base_allowed & ~allowed).sum())
        stats["gap_candidates"] += int(allowed.sum())
        if not allowed.any():
            continue

        big = float(np.max(adaptive)) * 1000.0 + 1.0
        row_index, col_index = linear_sum_assignment(np.where(allowed, distance, big))

        for r, c in zip(row_index, col_index):
            if not allowed[r, c]:
                continue
            source_id = end_ids[int(r)]
            target_id = start_ids[int(c)]
            if source_id in outgoing or target_id in used_starts:
                continue
            if not base_allowed[r, c]:
                stats["gap_density_selected_outside_base"] += 1

            source = nodes_by_id[source_id]
            target = nodes_by_id[target_id]
            mid_t = int(source["t"]) + 1
            mid_point = (
                (float(source["z"]) + float(target["z"])) / 2.0,
                (float(source["y"]) + float(target["y"])) / 2.0,
                (float(source["x"]) + float(target["x"])) / 2.0,
            )

            middle_id = None
            if GAP_REUSE_EXISTING:
                pool_ids, pool_coords, alive = isolated_pool(mid_t)
                if alive.any():
                    reuse = np.where(
                        alive,
                        np.linalg.norm(pool_coords - np.asarray(mid_point) * SCALE, axis=1),
                        np.inf,
                    )
                    best = int(np.argmin(reuse))
                    if reuse[best] <= GAP_REUSE_UM:
                        middle_id = int(pool_ids[best])
                        alive[best] = False
                        stats["gap_reused_existing"] += 1

            if middle_id is None:
                if synthetic_added >= max_synthetic:
                    stats["gap_skipped_node_cap"] += 1
                    continue
                middle_id = next_id
                next_id += 1
                refined = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)
                nodes_by_id[middle_id] = {
                    "node_id": middle_id,
                    "t": mid_t,
                    "z": refined[0],
                    "y": refined[1],
                    "x": refined[2],
                    "gap_synthetic": 1,
                }
                synthetic_added += 1
                stats["gap_inserted_synthetic"] += 1

            middle = nodes_by_id[middle_id]
            new_edges.append({
                "source_id": source_id,
                "target_id": middle_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(source, middle),
                "gap_closed": 1,
            })
            new_edges.append({
                "source_id": middle_id,
                "target_id": target_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(middle, target),
                "gap_closed": 1,
            })
            outgoing.update((source_id, middle_id))
            incoming.update((middle_id, target_id))
            used_starts.add(target_id)
            stats["gap_pairs_selected"] += 1
            stats["gap_added_edges"] += 2

    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, ([*edges, *new_edges] if new_edges else edges)


# ---------------------------------------------------------------------------
# Geometry-only division fallback (used for the anytime baseline shard)
# ---------------------------------------------------------------------------
def add_safe_divisions_postlink(nodes_by_id, edges, stats):
    if not SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges

    out_by_source: dict[int, list[dict]] = {}
    incoming: set[int] = set()
    for edge in edges:
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"]))

    ids_by_t = _ids_by_frame(nodes_by_id)
    existing_edges = {(int(e["source_id"]), int(e["target_id"])) for e in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict] = []
    used_targets: set[int] = set()

    for t in sorted(ids_by_t):
        if t + 1 not in ids_by_t:
            continue
        source_ids = [i for i in ids_by_t[t] if len(out_by_source.get(i, ())) == 1]
        candidate_ids = [
            i for i in ids_by_t[t + 1] if i not in incoming and i not in used_targets
        ]
        if not source_ids or not candidate_ids:
            continue

        candidate_coords = _coords_um(nodes_by_id, candidate_ids)
        candidate_tree = cKDTree(candidate_coords)
        source_coords = _coords_um(nodes_by_id, source_ids)
        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float]] = []

        for index, source_id in enumerate(source_ids):
            existing_child_id = int(out_by_source[source_id][0]["target_id"])
            existing_child = nodes_by_id.get(existing_child_id)
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            child_position = np.asarray(node_point(existing_child)) * SCALE
            if float(np.linalg.norm(child_position - source_coords[index])) > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
            near = candidate_tree.query_ball_point(source_coords[index], SAFE_DIV_MAX_UM)
            if not near:
                continue
            near = np.asarray(near, dtype=np.int64)
            parent_distance = np.linalg.norm(candidate_coords[near] - source_coords[index], axis=1)
            sister_distance = np.linalg.norm(candidate_coords[near] - child_position, axis=1)
            keep = (parent_distance <= SAFE_DIV_MAX_UM) & (sister_distance <= SAFE_DIV_SISTER_MAX_UM)
            for offset in np.flatnonzero(keep):
                candidate_id = candidate_ids[int(near[offset])]
                if (source_id, candidate_id) in existing_edges:
                    continue
                proposals.append((
                    float(parent_distance[offset])
                    + SAFE_DIV_SISTER_SCORE_WEIGHT * float(sister_distance[offset]),
                    source_id,
                    candidate_id,
                    float(parent_distance[offset]),
                ))

        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])
        added_this_frame = 0
        for _, source_id, candidate_id, parent_distance in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_distance,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            added_this_frame += 1

    if not added:
        return edges
    stats["safe_divisions_added"] = len(added)
    return [*edges, *added]


# ---------------------------------------------------------------------------
# Component filtering and trajectory smoothing
# ---------------------------------------------------------------------------
def filter_short_track_components(nodes_by_id, edges, stats):
    if not FILTER_SHORT_TRACKS or MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    out_count: dict[int, int] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        if source_id in parent and target_id in parent:
            root_a, root_b = find(source_id), find(target_id)
            if root_a != root_b:
                parent[root_a] = root_b
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    keep: set[int] = set()
    for members in components.values():
        divides = KEEP_DIVISION_COMPONENTS and any(out_count.get(i, 0) >= 2 for i in members)
        if len(members) >= MIN_TRACK_LEN or divides:
            keep.update(members)

    if not keep or len(keep) == len(nodes_by_id):
        return nodes_by_id, edges

    kept_nodes = {i: node for i, node in nodes_by_id.items() if i in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(
        1 for members in components.values() if not (set(members) & keep)
    )
    stats["short_track_nodes_removed"] = len(nodes_by_id) - len(kept_nodes)
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(nodes_by_id, edges, stats):
    """Blend each node towards a local linear fit; topology is untouched.

    Neighbourhoods only extend through unambiguous single links, so with a
    window of W there are at most (W + 1)^2 distinct shapes. Nodes are grouped
    by shape and each group is fitted in one batched least-squares.
    """
    if not LINEFIT_SMOOTH or LINEFIT_WEIGHT <= 0 or LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    # `None` marks "not exactly one link", which stops a walk just like a gap.
    predecessor: dict[int, int | None] = {}
    successor: dict[int, int | None] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None or int(target["t"]) != int(source["t"]) + 1:
            continue
        successor[source_id] = None if source_id in successor else target_id
        predecessor[target_id] = None if target_id in predecessor else source_id

    ids = sorted(nodes_by_id)
    index_of = {node_id: index for index, node_id in enumerate(ids)}
    coords = np.empty((len(ids), 3), dtype=np.float64)
    for index, node_id in enumerate(ids):
        node = nodes_by_id[node_id]
        coords[index] = (node["z"], node["y"], node["x"])

    shapes: dict[tuple[int, int], list[list[int]]] = {}
    for index, node_id in enumerate(ids):
        row = [index]
        counts = []
        for links in (predecessor, successor):
            current = node_id
            steps = 0
            while steps < LINEFIT_WINDOW:
                nearby = links.get(current)
                if nearby is None or nearby not in index_of:
                    break
                current = nearby
                row.append(index_of[current])
                steps += 1
            counts.append(steps)
        if len(row) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue
        shapes.setdefault((counts[0], counts[1]), []).append(row)

    weight = float(np.clip(LINEFIT_WEIGHT, 0.0, 1.0))
    # Every fit reads the original positions; results are applied at the end.
    smoothed = coords.copy()
    touched: list[np.ndarray] = []
    for (back, forward), rows in shapes.items():
        offsets = np.array(
            [0, *range(-1, -back - 1, -1), *range(1, forward + 1)], dtype=np.float64
        )
        centered = offsets - offsets.mean()
        denominator = float(centered @ centered)
        if denominator <= 0.0:
            stats["linefit_skipped_nodes"] += len(rows)
            continue
        members = np.asarray(rows, dtype=np.int64)
        window = coords[members]                                # (m, k, 3)
        mean = window.mean(axis=1)
        # Closed-form degree-1 least squares evaluated at offset 0.
        slope = np.einsum("k,mkc->mc", centered, window - mean[:, None, :]) / denominator
        fitted = mean - slope * offsets.mean()
        usable = np.isfinite(fitted).all(axis=1)
        stats["linefit_skipped_nodes"] += int((~usable).sum())
        targets = members[usable, 0]
        smoothed[targets] = (1.0 - weight) * coords[targets] + weight * fitted[usable]
        touched.append(targets)

    updated = np.concatenate(touched) if touched else np.empty(0, dtype=np.int64)
    for index in updated:
        node = nodes_by_id[ids[int(index)]]
        point = smoothed[int(index)]
        node["z"], node["y"], node["x"] = float(point[0]), float(point[1]), float(point[2])
    stats["linefit_smoothed_nodes"] = int(updated.size)
    return nodes_by_id


# ---------------------------------------------------------------------------
# Division decoder (Model C + division GBM + source-cardinality head)
# ---------------------------------------------------------------------------
DIVISION_RUNTIME = None


def _load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Unable to load {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def init_division_runtime():
    """Load the decoder once in the parent; forked workers inherit it."""
    global DIVISION_RUNTIME
    if DIVISION_RUNTIME is not None or not COMBINED_DECODER:
        return DIVISION_RUNTIME

    gbm = _load_module(
        "biohub_division_gbm_runtime", MODEL_C_DIR / "division_gbm_runtime.py"
    ).DivisionGBMRuntime(MODEL_C_DIR)
    legacy = _load_module(
        "biohub_model_c_safe_runtime", MODEL_C_DIR / "model_c_safe_runtime.py"
    ).CombinedModelCPrimaryRuntime(gbm, MODEL_C_DIR, threshold=MODEL_C_THRESHOLD)

    if not SOURCE_CARDINALITY_ENABLED:
        DIVISION_RUNTIME = legacy
        return DIVISION_RUNTIME

    if str(SOURCE_CARDINALITY_DIR) not in sys.path:
        sys.path.insert(0, str(SOURCE_CARDINALITY_DIR))
    DIVISION_RUNTIME = _load_module(
        "biohub_source_cardinality_graph_runtime",
        SOURCE_CARDINALITY_DIR / "source_cardinality_graph_runtime.py",
    ).SourceCardinalityGraphRuntime(
        gbm, SOURCE_CARDINALITY_DIR, MODEL_C_DIR, legacy_fallback_runtime=legacy
    )
    return DIVISION_RUNTIME


def _evidence(root: Path, dataset: str) -> Path:
    path = root / f"{dataset}.npz"
    if not path.exists():
        raise FileNotFoundError(f"Missing native evidence: {path}")
    return path


def apply_division_decoder(dataset, nodes_by_id, edges, stats):
    if DIVISION_RUNTIME is None:
        init_division_runtime()
    return DIVISION_RUNTIME.apply(
        TEST_DIR / f"{dataset}.zarr",
        _evidence(MODEL_C_EVIDENCE_DIR, dataset),
        _evidence(P1_EVIDENCE_DIR, dataset),
        _evidence(P2_EVIDENCE_DIR, dataset),
        nodes_by_id,
        edges,
        stats,
        spacing=VOXEL_SCALE_UM,
        v2_threshold=GBM_THRESHOLD,
        v2_rescue_delta=GBM_RESCUE_DELTA,
        v2_steal_delta=GBM_STEAL_DELTA,
    )


# ---------------------------------------------------------------------------
# Full graph filter
# ---------------------------------------------------------------------------
def filter_output_graph(nodes_by_id, raw_edges, dataset=None, division_mode="combined"):
    stats = defaultdict(int, BASE_STATS)
    stats["raw_edges"] = len(raw_edges)

    edges: list[dict] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"]))
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue
        if ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        distance = edge_distance_um(source, target)
        edge["distance_um"] = distance
        if EDGE_MAX_UM > 0 and distance > EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge)

    if MOTION_RELINK:
        motion_edges = motion_relink_edges(nodes_by_id, stats, _sanitized_edge_probs(edges))
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1

    if SINGLE_PARENT_REPAIR and edges:
        best_by_target: dict[int, dict] = {}
        for edge in edges:
            target_id = int(edge["target_id"])
            previous = best_by_target.get(target_id)
            if previous is None or edge_sort_key(edge) > edge_sort_key(previous):
                best_by_target[target_id] = edge
        kept = {id(edge) for edge in best_by_target.values()}
        stats["dropped_multi_parent_edges"] = len(edges) - len(kept)
        edges = [edge for edge in edges if id(edge) in kept]

    nodes_by_id, edges = close_single_frame_gaps(
        nodes_by_id, edges, stats, dataset=dataset, frame_cache={}
    )

    if division_mode == "combined" and COMBINED_DECODER:
        if dataset is None:
            raise RuntimeError("The combined decoder needs a dataset id")
        edges = apply_division_decoder(dataset, nodes_by_id, edges, stats)
    else:
        edges = add_safe_divisions_postlink(nodes_by_id, edges, stats)

    if PRUNE_ISOLATED:
        incident = {int(e["source_id"]) for e in edges} | {int(e["target_id"]) for e in edges}
        if incident:
            kept_nodes = {i: node for i, node in nodes_by_id.items() if i in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [
                edge for edge in edges
                if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id
            ]

    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)
    return nodes_by_id, edges, dict(stats)


# ---------------------------------------------------------------------------
# geff -> submission shard
# ---------------------------------------------------------------------------
def graph_nodes_edges(path: Path):
    import tracksdata as td

    graph = td.graph.IndexedRXGraph.from_geff(path)
    if isinstance(graph, tuple):
        graph = graph[0]

    nodes: dict[int, dict] = {}
    for row in graph.node_attrs().iter_rows(named=True):
        node_id = int(row["node_id"])
        nodes[node_id] = {
            "node_id": node_id,
            "t": int(row["t"]),
            "z": float(row["z"]),
            "y": float(row["y"]),
            "x": float(row["x"]),
        }
    edges = []
    for row in graph.edge_attrs().iter_rows(named=True):
        prob = row.get("edge_prob")
        edges.append({
            "source_id": int(row["source_id"]),
            "target_id": int(row["target_id"]),
            "edge_prob": None if prob is None else float(prob),
        })
    return nodes, edges


def write_dataset_shard(dataset: str, nodes_by_id: dict, edges: list[dict]):
    destination = SHARD_DIR / f"{dataset}.csv"
    temporary = destination.with_suffix(".csv.tmp")
    row_id = 0
    out_degree: dict[int, int] = {}
    with temporary.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=CSV_COLUMNS)
        writer.writeheader()
        for node_id in sorted(nodes_by_id):
            node = nodes_by_id[node_id]
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(node["node_id"]),
                "t": int(node["t"]),
                "z": max(0, int(round(float(node["z"])))),
                "y": max(0, int(round(float(node["y"])))),
                "x": max(0, int(round(float(node["x"])))),
                "source_id": -1,
                "target_id": -1,
            })
            row_id += 1
        for edge in edges:
            source_id = int(edge["source_id"])
            target_id = int(edge["target_id"])
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": source_id,
                "target_id": target_id,
            })
            row_id += 1
            out_degree[source_id] = out_degree.get(source_id, 0) + 1
    temporary.replace(destination)
    return len(nodes_by_id), len(edges), sum(v >= 2 for v in out_degree.values())


def process_graph(geff_path: Path, division_mode: str) -> dict:
    """Post-process one prediction graph and atomically replace its shard."""
    import time

    dataset = Path(geff_path).stem
    nodes_by_id, raw_edges = graph_nodes_edges(Path(geff_path))
    raw_nodes = len(nodes_by_id)

    started = time.monotonic()
    nodes_by_id, edges, stats = filter_output_graph(
        nodes_by_id, raw_edges, dataset=dataset, division_mode=division_mode
    )
    seconds = time.monotonic() - started
    if not nodes_by_id:
        raise AssertionError(f"{dataset}: post-processing removed every node")

    node_count, edge_count, divisions = write_dataset_shard(dataset, nodes_by_id, edges)
    stats.update({
        "dataset": dataset,
        "raw_nodes": raw_nodes,
        "nodes": node_count,
        "edges": edge_count,
        "division_like_sources": divisions,
        "edge_to_node_ratio": edge_count / max(node_count, 1),
        "division_mode": division_mode,
        "process_seconds": seconds,
    })
    return stats

## Streaming schedule

The GPU producers emit one `.geff` graph plus its Model-C / P1 / P2 evidence per
video and mark it with an atomic `.ready` file. As soon as a marker appears the
main process writes a **baseline** shard (geometry-only divisions), then hands the
same graph to a CPU worker for the **combined** upgrade (Model C + division GBM +
source-cardinality head). A shard is replaced only after a full successful
upgrade, so the submission is valid at every point in time and degrades to the
baseline if the 11 h 50 m budget runs out.

In [ ]:
"""Streaming inference: GPU producers -> anytime baseline -> CPU upgrades.

Each video is written as a self-contained CSV shard. A shard is replaced only
after a full successful upgrade, so the run is submission-safe at any moment.
"""
import csv
import multiprocessing as mp
import queue as stdlib_queue
import signal
import subprocess
import time
import traceback

import pandas as pd
from threadpoolctl import threadpool_limits

if os.name != "posix":
    raise RuntimeError("The forked worker pool requires Kaggle's POSIX runtime")

sys.path.insert(0, str(PKG_DIR))
import postprocess as pp

pp.init_division_runtime()

RUNTIME = CFG["runtime"]
METHOD = "unet_transformer"
SHARD_DIR = Path(CFG["paths"]["shard_dir"])
DEADLINE = (
    NOTEBOOK_START + RUNTIME["hard_limit_seconds"] - RUNTIME["finalize_reserve_seconds"]
)

# --------------------------------------------------------------------------
# GPU producers
# --------------------------------------------------------------------------
splits_path = REPO_DIR / "kaggle_test_splits.json"
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": TEST_STEMS}]))

predict_cmd = [
    sys.executable, "scripts/predict_unet_transformer.py",
    "--data-dir", str(TEST_DIR),
    "--splits", splits_path.name,
    "--split", "0",
    "--weights", P1_WEIGHTS_REL,
    "--unet-batch-size", str(CFG["detection"]["unet_batch_size"]),
    "--det-threshold", str(CFG["detection"]["threshold"]),
    "--ilp-edge-weight", str(CFG["ilp"]["edge_weight"]),
    "--ilp-appearance-weight", str(CFG["ilp"]["appearance"]),
    "--ilp-disappearance-weight", str(CFG["ilp"]["disappearance"]),
    "--ilp-division-weight", str(CFG["ilp"]["division"]),
]
if CFG["ilp"]["use"]:
    predict_cmd.append("--use-ilp")
slice_spec = str(RUNTIME["slice"]).strip()
if slice_spec:
    predict_cmd += ["--slice", slice_spec]

visible = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()
cuda_tokens = (
    [token.strip() for token in visible.split(",") if token.strip()]
    if visible and visible != "-1"
    else [str(index) for index in range(torch.cuda.device_count())]
)
shard_count = min(RUNTIME["gpu_workers"], len(cuda_tokens), len(TEST_STEMS))
sharded = shard_count >= 2 and not slice_spec
helper_deadline_epoch = time.time() + max(0.0, DEADLINE - time.monotonic())

predict_processes: dict[int, subprocess.Popen] = {}
predict_commands: dict[int, list[str]] = {}
predict_methods: list[str] = []
predict_started = time.monotonic()

for index in range(shard_count if sharded else 1):
    method = f"{METHOD}_gpu{index}" if sharded else METHOD
    command = list(predict_cmd)
    if sharded:
        command += ["--method", method, "--slice", f"{index}::{shard_count}"]
    environment = {
        **os.environ,
        "PYTHONPATH": "src",
        "CUDA_VISIBLE_DEVICES": cuda_tokens[index],
        "BIOHUB_HELPER_DEADLINE_EPOCH": str(helper_deadline_epoch),
    }
    predict_methods.append(method)
    predict_commands[index] = command
    predict_processes[index] = subprocess.Popen(command, cwd=REPO_DIR, env=environment)

print(f"GPU producers: {len(predict_processes)} on CUDA {cuda_tokens[:len(predict_processes)]}")

# --------------------------------------------------------------------------
# CPU upgrade workers
# --------------------------------------------------------------------------
CPU_WORKERS = max(1, int(RUNTIME["cpu_workers"]))
CPU_WORKERS_WHILE_PREDICTING = max(1, min(int(RUNTIME["cpu_workers_while_predicting"]), CPU_WORKERS))


class _UpgradeDeadline(TimeoutError):
    pass


def _on_alarm(signum, frame):
    raise _UpgradeDeadline("upgrade reached the submission-safe deadline")


def _worker_main(worker_id: int, task_queue, result_queue) -> None:
    for variable in (
        "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
        "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS",
    ):
        os.environ[variable] = "1"
    signal.signal(signal.SIGALRM, _on_alarm)
    with threadpool_limits(limits=1):
        while True:
            task = task_queue.get()
            if task is None:
                return
            dataset, graph_path = task
            started = time.monotonic()
            remaining = DEADLINE - started
            if remaining <= 0:
                result_queue.put({"worker_id": worker_id, "dataset": dataset,
                                  "status": "deadline", "error": "no time left"})
                continue
            signal.setitimer(signal.ITIMER_REAL, max(remaining, 1.0))
            try:
                row = pp.process_graph(Path(graph_path), "combined")
                result_queue.put({"worker_id": worker_id, "dataset": dataset,
                                  "status": "ok", "seconds": time.monotonic() - started,
                                  "row": row})
            except _UpgradeDeadline as error:
                result_queue.put({"worker_id": worker_id, "dataset": dataset,
                                  "status": "deadline", "error": str(error)})
            except BaseException as error:
                result_queue.put({"worker_id": worker_id, "dataset": dataset,
                                  "status": "error", "error": f"{type(error).__name__}: {error}",
                                  "traceback": traceback.format_exc(limit=12)})
            finally:
                signal.setitimer(signal.ITIMER_REAL, 0.0)


context = mp.get_context("fork")
result_queue = context.Queue()
task_queues = [context.Queue(maxsize=1) for _ in range(CPU_WORKERS)]
workers = []
for worker_id in range(CPU_WORKERS):
    process = context.Process(
        target=_worker_main,
        args=(worker_id, task_queues[worker_id], result_queue),
        name=f"biohub-upgrade-{worker_id}",
        daemon=False,
    )
    process.start()
    workers.append(process)
worker_task: dict[int, str | None] = {worker_id: None for worker_id in range(CPU_WORKERS)}
print(f"CPU upgrade workers: {CPU_WORKERS} ({CPU_WORKERS_WHILE_PREDICTING} while the GPU runs)")

# --------------------------------------------------------------------------
# Scheduler
# --------------------------------------------------------------------------
stats_by_dataset: dict[str, dict] = {}
graph_by_dataset: dict[str, Path] = {}
baseline_done: set[str] = set()
attempted: set[str] = set()
upgraded: list[str] = []
evidence_roots = [Path(CFG["paths"][key]) for key in
                  ("model_c_evidence_dir", "p1_evidence_dir", "p2_evidence_dir")]


def _ready_graphs() -> dict[str, Path]:
    ready = {}
    for method in predict_methods:
        for marker in (REPO_DIR / "predictions").glob(f"*/{method}/split_0/*.ready"):
            graph = marker.with_suffix(".geff")
            if graph.exists() and marker.stem in TEST_STEMS:
                ready[marker.stem] = graph
    return ready


def _collect_results() -> None:
    while True:
        try:
            result = result_queue.get_nowait()
        except stdlib_queue.Empty:
            return
        worker_task[int(result["worker_id"])] = None
        dataset = str(result["dataset"])
        if result["status"] == "ok":
            stats_by_dataset[dataset] = result["row"]
            upgraded.append(dataset)
            print(f"[upgrade {len(upgraded):3d}] {dataset}: "
                  f"{result['seconds'] / 60:.1f}m "
                  f"wall={(time.monotonic() - NOTEBOOK_START) / 3600:.2f}h", flush=True)
        else:
            print(f"[fallback] {dataset}: {result['status']} {result.get('error', '')}",
                  flush=True)
            if result.get("traceback"):
                print(result["traceback"], flush=True)


producer_finished_at = None
while True:
    ready = _ready_graphs()
    graph_by_dataset.update(ready)
    for dataset in sorted(set(ready) - baseline_done):
        stats_by_dataset[dataset] = pp.process_graph(ready[dataset], "baseline")
        baseline_done.add(dataset)
        print(f"[baseline {len(baseline_done):3d}/{len(TEST_STEMS)}] {dataset}: "
              f"nodes={stats_by_dataset[dataset]['nodes']:,} "
              f"edges={stats_by_dataset[dataset]['edges']:,}", flush=True)

    _collect_results()
    for worker_id, process in enumerate(workers):
        if process.exitcode is not None and worker_task[worker_id] is not None:
            print(f"[worker exit] {worker_task[worker_id]} keeps its baseline shard", flush=True)
            worker_task[worker_id] = None

    producer_done = all(process.poll() is not None for process in predict_processes.values())
    if producer_done and producer_finished_at is None:
        producer_finished_at = time.monotonic()
    remaining = DEADLINE - time.monotonic()

    evidence_ready = {
        dataset for dataset in baseline_done
        if all((root / f"{dataset}.npz").exists() for root in evidence_roots)
    }
    pending = sorted(
        (graph_by_dataset[dataset] for dataset in evidence_ready - attempted),
        key=lambda path: (int(stats_by_dataset[path.stem].get("raw_nodes", 0)), path.stem),
    )
    if remaining > RUNTIME["min_upgrade_seconds"] and pending:
        active_limit = CPU_WORKERS if producer_done else CPU_WORKERS_WHILE_PREDICTING
        idle = [
            worker_id for worker_id in range(active_limit)
            if worker_task[worker_id] is None and workers[worker_id].is_alive()
        ]
        for worker_id, graph in zip(idle, pending):
            attempted.add(graph.stem)
            worker_task[worker_id] = graph.stem
            task_queues[worker_id].put((graph.stem, str(graph)))

    busy = any(dataset is not None for dataset in worker_task.values())
    all_baselines = len(baseline_done) == len(TEST_STEMS)
    if producer_done and all_baselines and not busy and not (evidence_ready - attempted):
        break
    if remaining <= 0 and producer_done and all_baselines and not busy:
        break
    time.sleep(2.0)

_collect_results()
for worker_id, task_queue in enumerate(task_queues):
    if workers[worker_id].is_alive():
        task_queue.put(None)
for process in workers:
    process.join(timeout=10)
    if process.is_alive():
        process.terminate()
        process.join(timeout=5)

for index, process in predict_processes.items():
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, predict_commands[index])

missing = sorted(set(TEST_STEMS) - baseline_done)
if missing:
    raise RuntimeError(f"Producer finished but shards are missing: {missing[:10]}")

# --------------------------------------------------------------------------
# Submission
# --------------------------------------------------------------------------
row_id = node_rows = edge_rows = 0
with SUBMISSION_PATH.open("w", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=pp.CSV_COLUMNS)
    writer.writeheader()
    for dataset in sorted(TEST_STEMS):
        shard = SHARD_DIR / f"{dataset}.csv"
        if not shard.exists():
            raise FileNotFoundError(f"Missing shard: {shard}")
        with shard.open(newline="") as handle:
            for row in csv.DictReader(handle):
                row["id"] = row_id
                writer.writerow(row)
                row_id += 1
                node_rows += row["row_type"] == "node"
                edge_rows += row["row_type"] == "edge"

assert node_rows > 0 and row_id == node_rows + edge_rows

predict_hours = ((producer_finished_at or time.monotonic()) - predict_started) / 3600.0
stats = pd.DataFrame(
    [stats_by_dataset[dataset] for dataset in sorted(stats_by_dataset)]
).sort_values("dataset").reset_index(drop=True)
stats["experiment_tag"] = CFG["experiment_tag"]
stats["predict_hours"] = predict_hours
stats["upgraded_total"] = len(upgraded)
stats["wall_hours"] = (time.monotonic() - NOTEBOOK_START) / 3600.0
stats.to_csv(RUN_STATS_PATH, index=False)

print(f"\n{SUBMISSION_PATH}: {row_id:,} rows ({node_rows:,} nodes, {edge_rows:,} edges)")
print(f"Combined-decoder upgrades: {len(upgraded)}/{len(TEST_STEMS)} "
      f"| GPU {predict_hours:.2f}h | wall {(time.monotonic() - NOTEBOOK_START) / 3600:.2f}h")

In [ ]:
"""Structural checks on the written submission."""
submission = pd.read_csv(SUBMISSION_PATH)
if list(submission.columns) != pp.CSV_COLUMNS:
    raise AssertionError(f"Unexpected columns: {list(submission.columns)}")

nodes = submission[submission["row_type"] == "node"]
edges = submission[submission["row_type"] == "edge"]
if sorted(nodes["dataset"].unique()) != sorted(TEST_STEMS):
    raise AssertionError("Submission does not cover every test video")
if nodes.duplicated(["dataset", "node_id"]).any():
    raise AssertionError("Duplicate node ids inside a dataset")

frame_of = {
    (dataset, node_id): t
    for dataset, node_id, t in zip(nodes["dataset"], nodes["node_id"], nodes["t"])
}
bad_step = in_degree = out_degree = 0
parents: dict[tuple[str, int], int] = {}
children: dict[tuple[str, int], int] = {}
for dataset, source_id, target_id in zip(edges["dataset"], edges["source_id"], edges["target_id"]):
    source_t = frame_of.get((dataset, source_id))
    target_t = frame_of.get((dataset, target_id))
    if source_t is None or target_t is None or target_t != source_t + 1:
        bad_step += 1
    parents[(dataset, target_id)] = parents.get((dataset, target_id), 0) + 1
    children[(dataset, source_id)] = children.get((dataset, source_id), 0) + 1

in_degree = sum(1 for count in parents.values() if count > 1)
out_degree = sum(1 for count in children.values() if count > 2)
divisions = sum(1 for count in children.values() if count == 2)
if bad_step or in_degree or out_degree:
    raise AssertionError(
        f"Invalid lineage: bad_step={bad_step} multi_parent={in_degree} over_two_children={out_degree}"
    )

print(f"OK | videos={nodes['dataset'].nunique()} nodes={len(nodes):,} "
      f"edges={len(edges):,} divisions={divisions:,}")
display(submission.head(5))